In [ ]:
#==================  Cq and GF Combined - Cell-1  ======================
# Cell-1: Collect user inputs for qPCR Analysis (Cq and GF Combined)
"""
Purpose:
- Collect user inputs for the qPCR analysis pipeline to produce both Cq and Global-fitting (GF) outputs.
- Gather the path to the qPCR data file (supports root directory filenames in Colab, e.g., 'qpcr_data.csv').
- Assume input data are "raw" (preferred: unadjusted, not platform-corrected).
- Assign optional output flags: Evaluation (-e) and Debug (-d).
- Validate inputs with explanations in prompts and provide retry loops for error recovery (up to 3 attempts per input).
- Create outputs directory for generated files.
- Store validated inputs for use in subsequent cells.

Default Behavior (press Enter for each flag prompt):
- Evaluation (-e) = False: No extra outputs beyond the final report
- Debug (-d) = False: No intermediate files/plots, only final CSV from Cell-11

Optional Flags:
- Enter -e: Evaluation mode (limited set of helpful outputs for inspection)
- Enter -d: Debug mode (extensive set of intermediate data and graphical outputs)

Outputs:
- file_path: String, path to the qPCR data file.
- eval_flag: True or False, indicating whether to generate evaluation outputs (default: False).
- debug_flag: True or False, indicating whether to generate debug outputs (default: False).
- debug_display_flag: True or False, indicating whether to display debug plots (default: False).

- output_dir: String, path to the outputs directory.
"""

import sys
import os

def get_input_with_retry(prompt, validator=None, max_retries=3):
    """
    Helper function to get input with retries on invalid responses.
    - prompt: String to display to user.
    - validator: Optional function that takes input and returns True if valid, False otherwise.
    - max_retries: Max attempts before raising error.
    """
    for attempt in range(max_retries):
        user_input = input(prompt).strip()
        if validator is None or validator(user_input):
            return user_input
        print(f"Invalid input. Please try again. (Attempt {attempt + 1}/{max_retries})")
    raise ValueError(f"Max retries exceeded for input: {prompt}")

# Create outputs directory
output_dir = "outputs"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created outputs directory: {output_dir}")
else:
    print(f"Using existing outputs directory: {output_dir}")

# Collect and validate file path with retry
file_prompt = "Enter the path to your raw qPCR data file (e.g., 'qpcr_data.csv' for root directory in Colab, or full path like '/content/qpcr_data.csv'):\n"
file_path = get_input_with_retry(file_prompt, lambda fp: os.path.isfile(fp))

# Collect evaluation flag (-e)
eval_prompt = "Enter -e for Evaluation output (limited key files + plots), or press Enter for none [default: none]:\n"
eval_input = get_input_with_retry(eval_prompt, lambda ei: ei.lower() in ['-e', ''] or ei == '').lower()
eval_flag = True if eval_input == '-e' else False

# Collect debug flag (-d)
debug_prompt = "Enter -d for Debug output (full intermediate files + plots), or press Enter for minimal output [default: minimal]:\n"
debug_input = get_input_with_retry(debug_prompt, lambda di: di.lower() in ['-d', ''] or di == '').lower()
debug_flag = True if debug_input == '-d' else False

# Debug display flag (intermediate plots shown only in Debug)
debug_display_flag = debug_flag

# Print collected inputs for confirmation
print("\nCollected Inputs:")
print(f"File path: {file_path}")
print(f"Evaluation flag: {eval_flag}")
print(f"Debug flag: {debug_flag}")
print(f"Output directory: {output_dir}")
print("Note: Input data are assumed to be raw (unadjusted, not machine-corrected).")

# Outputs: file_path, debug_flag, debug_display_flag, eval_flag, output_dir


In [ ]:
#==================  Cq and GF Combined - Cell-2   ======================
# Cell-2: Load and prepare qPCR data for qPCR analysis
"""
Purpose:
- Load raw qPCR data from the specified file path (raw data preferred, not machine-corrected input).
- Validate the data format, headers, and data entries.
- Identify columns containing fluorescence data, excluding metadata columns.
- Set the DataFrame index to the 'Cycle' column (if present) or adjust to start at 1.
- Provide a preview of the first 5 data points with the index labeled as 'Cycle' and a summary of maximum fluorescence.
- Plot the raw fluorescence data for all specified columns to visualize the dataset.
- Add detailed debugging output to inspect the loaded data and validation results (Debug mode).

Inputs:
- file_path: String, path to the qPCR data file (from Cell-1).
- eval_flag: True or False, indicating Evaluation mode (from Cell-1).
- debug_flag: True or False, indicating Debug mode (from Cell-1).
- debug_display_flag: True or False, indicating Debug plot display (from Cell-1).
- output_dir: String, path to the outputs directory (from Cell-1).

Outputs:
- df: DataFrame containing the qPCR fluorescence data, with index set to cycles starting at 1.
- columns_to_fit: List of column names with fluorescence data to process in subsequent cells.
- (Optional, Debug): Save the raw data plot as a PNG file.
"""

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import datetime

# Verify inputs from Cell-1
try:
    file_path, debug_flag, debug_display_flag, eval_flag, output_dir
except NameError:
    raise NameError("Required variables (file_path, debug_flag, debug_display_flag, eval_flag, output_dir) not defined. Please run Cell-1 first.")

# Load the data
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    raise FileNotFoundError(f"Data file not found at {file_path}. Please check the file path and ensure it exists.")
except Exception as e:
    raise Exception(f"Error loading data file: {e}")

# Validate the DataFrame
if df.empty:
    raise ValueError("Loaded DataFrame is empty. Please check the data file and ensure it contains data.")

# Set the index to the 'Cycle' column if present, otherwise adjust the default index
cycle_column = None
for col in df.columns:
    if col.lower() == 'cycle':
        cycle_column = col
        break

if cycle_column is not None:
    try:
        df[cycle_column] = pd.to_numeric(df[cycle_column], errors='coerce')
        if df[cycle_column].isna().any():
            raise ValueError(f"Cycle column '{cycle_column}' contains non-numeric values. Please clean the data.")
        if df[cycle_column].min() != 1:
            print(f"Warning: Cycle column '{cycle_column}' does not start at 1 (min={df[cycle_column].min()}). Adjusting cycles to start at 1.")
            df[cycle_column] = df[cycle_column] - df[cycle_column].min() + 1
        df.set_index(cycle_column, inplace=True)
    except Exception as e:
        raise ValueError(f"Error setting index to Cycle column: {e}")
else:
    df.index = range(1, len(df) + 1)
    print("Warning: No 'Cycle' column found. Using default index starting at 1.")
df.index.name = "Cycle"

# Validate column headers
unnamed_columns = [col for col in df.columns if col.startswith('Unnamed:')]
if unnamed_columns:
    raise ValueError(f"Found columns without headers: {unnamed_columns}. Please provide proper column names.")

# Identify potential metadata columns
metadata_columns = ['cycle', 'index', 'time', 'well', 'sample']
potential_metadata = [col for col in df.columns if col.lower() in metadata_columns]

# Identify fluorescence data columns
columns_to_fit = []
invalid_columns = []
for col in df.columns:
    if col.lower() in metadata_columns:
        continue
    try:
        numeric_series = pd.to_numeric(df[col], errors='coerce')
        if numeric_series.isna().any():
            invalid_rows = df.index[numeric_series.isna()].tolist()
            invalid_columns.append((col, invalid_rows))
        else:
            columns_to_fit.append(col)
    except Exception as e:
        invalid_columns.append((col, f"Error: {e}"))

if invalid_columns:
    error_msg = "Found non-numeric values or errors in the following columns:\n"
    for col, details in invalid_columns:
        error_msg += f" - {col}: {details}\n"
    raise ValueError(error_msg)

if not columns_to_fit:
    raise ValueError("No valid numeric columns found for fluorescence data. Please check the data file for numeric fluorescence values.")

# Check for missing data
if df[columns_to_fit].isna().any().any():
    raise ValueError("Data contains missing values (NaN). Please clean the dataset or replace missing values.")

# Preview: Print the first 5 rows and maximum fluorescence
print("\nFirst 5 rows of loaded data:")
print(df[columns_to_fit].head().to_string(index=True))
print("\nMaximum Fluorescence per Sample (Across All Rows):")
print(df[columns_to_fit].max().round(2).to_string())

# Plot the raw fluorescence data
def plot_raw_data(df, columns, debug_flag=False, debug_display_flag=False, output_dir="outputs", file_path=""):
    """
    Plot the raw fluorescence data for all specified columns.

    Args:
        df (pd.DataFrame): DataFrame with fluorescence data.
        columns (list): List of column names to plot.
        debug_flag (bool): Save debug outputs when True.
        debug_display_flag (bool): Display debug plots when True.
        output_dir (str): Directory to save plots.
        file_path (str): Original file path for naming saved plots.
    """
    if not columns:
        print("No columns to plot.")
        return

    if not debug_display_flag and not debug_flag:
        print("Debug output disabled. Skipping plot generation.")
        return

    fig_size = (12, 8) if len(columns) <= 10 else (15, 10)
    plt.figure(figsize=fig_size, constrained_layout=True)
    cycles = df.index.values  # Use the DataFrame index as cycles
    colors = plt.cm.tab10(np.linspace(0, 1, len(columns)))

    for idx, column in enumerate(columns):
        plt.plot(cycles, df[column], label=column, marker='o', linestyle='-', color=colors[idx])

    plt.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    plt.xlabel('Cycle')
    plt.ylabel('Fluorescence Value')
    plt.title('Raw Fluorescence Data: All Cycles')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2 if len(columns) > 10 else 1)
    plt.grid(True)

    if debug_flag:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        input_base_name = os.path.basename(file_path).replace('.csv', '')
        plot_path = os.path.join(output_dir, f'raw_data_plot_{input_base_name}_{timestamp}.png')
        plt.savefig(plot_path, bbox_inches='tight')
        print(f"Saved (Debug) raw data plot to {plot_path}")

    if debug_display_flag:
        plt.show()
        print("Plot displayed successfully.")
    else:
        plt.close()

# Plot the raw data
if columns_to_fit:
    plot_raw_data(df, columns_to_fit, debug_flag, debug_display_flag, output_dir, file_path)
else:
    print("No columns to plot.")

# Outputs: df, columns_to_fit


In [ ]:
#==================  Cq and GF Combined - Cell-3  ======================
# Cell-3: Estimate and subtract an initial background signal with x^n fitting and dynamic linear fallback
"""
Purpose:
- Estimate and subtract background signal from raw qPCR data to extrapolate to 0.0 at cycle 0.
- Fit F(c) = A * x^c + B starting with cycles 2-7 (to avoid cycle 1 aberrations) for each column.
- If exponential fit fails or efficiency (x) < 1.2, fallback to linear regression with dynamic extension:
  - Start with cycles 2-7, then iteratively add cycles if the new point's deviation is within 2-sigma of the current residuals' variance.
  - Revert inclusion of a 2-sigma outlier if the next point deviates further in the same direction, stopping extension.
  - Apply shift-only or shift+tilt based on the APPLY_BASELINE_TILT toggle.
- Report unadjusted and adjusted values for cycle 1.
- Plot 1: Original vs background-subtracted data for the first 10 cycles.
- Plot 2: Background-subtracted data only, across all cycles.
- Generate detailed reports in Debug mode.
- Prepare data for the next cell.

Inputs:
- df: DataFrame containing the qPCR fluorescence data (from Cell-2).
- columns_to_fit: List of column names to process (from Cell-2).
- debug_flag: True or False, indicating Debug mode (from Cell-1).
- debug_display_flag: True or False, indicating Debug plot display (from Cell-1).
- eval_flag: True or False, indicating Evaluation mode (from Cell-1).
- output_dir: String, path to the outputs directory (from Cell-1).
- file_path: String, path to the qPCR data file (from Cell-1).

Outputs:
- df_adjusted: DataFrame containing the background-subtracted fluorescence data, extrapolated to 0.0 at cycle 0.
- initial_backgrounds: Dictionary of estimated background values (extrapolated F(0) per column before adjustment).
- adjustment_types: Dictionary mapping each column to its adjustment type (exponential, linear_shift, linear_tilt).
- (Optional, Debug): Intermediate files (e.g., adjusted data CSV, background log, plot PNGs, cycle 1 comparison CSV).
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import os
import pandas as pd
import datetime
from scipy.stats import linregress

# Baseline adjustment option:
# True  = shift and tilt (level baseline by removing slope)
# False = shift only (subtract intercept)
APPLY_BASELINE_TILT = True

# Verify inputs from Cell-2
try:
    df, columns_to_fit
except NameError:
    raise NameError("Required variables (df, columns_to_fit) not defined. Please run Cell-2 first.")

# Verify inputs from Cell-1
try:
    debug_flag, debug_display_flag, eval_flag, output_dir, file_path
except NameError:
    raise NameError("Required variables (debug_flag, debug_display_flag, eval_flag, output_dir, file_path) not defined. Please run Cell-1 first.")

def general_exponential_func(c, A, x, B):
    """General exponential function for fitting: F(c) = A * x^c + B"""
    return A * (x ** c) + B

def estimate_background(data):
    """
    Estimate background signal starting with cycles 2-6, using exponential fit or dynamic linear fallback.
    - Exponential: Fit cycles 2-6, use if x > 1.2.
    - Linear: Start with cycles 2-6, extend dynamically if new cycle's deviation is within 2-sigma of residuals' variance,
              revert if next point trends further outward.
    Returns background (F(0)), adjustment type, and fit parameters.
    """
    total_cycles = len(data)
    initial_cycles = 6  # Start with 6 cycles (2-7)
    cycles = np.arange(2, min(8, total_cycles + 1))  # Initial window: cycles 2-7
    early_data = data[1:7]  # Indices 1-6 correspond to cycles 2-7

    try:
        initial_A = early_data[0] / 2  # Rough estimate assuming x = 2
        popt, _ = curve_fit(general_exponential_func, cycles, early_data, p0=[initial_A, 2.0, 0.0], maxfev=10000, bounds=(0, [np.inf, 2.5, np.inf]))
        A, x, B = popt
        background = A + B  # F(0) = A * x^0 + B
        if x < 1.2:
            print(f"Notice: Exponential fit x ({x:.4f}) < 1.2, falling back to linear regression.")
            raise RuntimeError("x < 1.2 detected")

        # Check if the exponential fit is meaningful (R² threshold)
        fitted_values = general_exponential_func(cycles, A, x, B)
        ss_res = np.sum((early_data - fitted_values)**2)
        ss_tot = np.sum((early_data - np.mean(early_data))**2)
        r_squared = 1 - (ss_res / ss_tot)
        print(f"Debug: SS_res = {ss_res:.4f}, SS_tot = {ss_tot:.4f}, R² = {r_squared:.4f}")
        print(f"Debug: early_data = {early_data}")
        print(f"Debug: fitted_values = {fitted_values}")
        if r_squared < 0.7:  # Low R² indicates poor fit
            print(f"Notice: Exponential fit R² ({r_squared:.4f}) too low, falling back to linear regression.")
            raise RuntimeError("Poor exponential fit detected")

        return background, 'exponential', {'A': A, 'x': x, 'B': B, 'cycles': cycles.tolist(), 'r_squared': r_squared}
    except (RuntimeError, ValueError) as e:
        print(f"Notice: Exponential fit failed ({e}). Falling back to linear regression.")

    # Dynamic linear extension
    current_cycles = cycles.copy()
    current_data = early_data.copy()
    residuals = np.array([])
    while len(current_cycles) < total_cycles:
        result = linregress(current_cycles, current_data)
        slope, intercept = result.slope, result.intercept
        fitted_values = slope * current_cycles + intercept
        residuals = current_data - fitted_values
        variance = np.var(residuals) if len(residuals) > 1 else 0.0
        sigma = np.sqrt(variance) * 2 if variance > 0 else 0.0  # 2-sigma threshold

        next_cycle = len(current_cycles) + 2  # Next cycle to consider (e.g., 8 after 2-7)
        if next_cycle > total_cycles:
            break
        next_value = data[next_cycle - 1]  # Index 6 for cycle 7, etc.
        predicted_value = slope * next_cycle + intercept
        deviation = abs(next_value - predicted_value)

        if len(residuals) > 0 and deviation <= sigma:  # Within 2-sigma
            last_deviation = residuals[-1] if len(residuals) > 0 else 0
            if len(current_data) > 1 and next_value > current_data[-1] and last_deviation > 0:  # Trending upward
                break  # Stop if next value trends further from mean
            elif len(current_data) > 1 and next_value < current_data[-1] and last_deviation < 0:  # Trending downward
                break  # Stop if next value trends further from mean
            current_cycles = np.append(current_cycles, next_cycle)
            current_data = np.append(current_data, next_value)
        else:
            break  # Stop if outside 2-sigma

    # Final linear fit
    result = linregress(current_cycles, current_data)
    slope, intercept = result.slope, result.intercept
    background = intercept  # Intercept at cycle 0
    if APPLY_BASELINE_TILT:
        adjustment_type = 'linear_tilt'
    else:
        adjustment_type = 'linear_shift'
    return background, adjustment_type, {'slope': slope, 'intercept': intercept, 'early_cycles': len(current_cycles), 'cycles': current_cycles.tolist()}

# Create a copy of the DataFrame for adjusted data
df_adjusted = df.copy()
initial_backgrounds = {}
adjustment_types = {}

# Apply background subtraction to extrapolate to 0.0 at cycle 0
background_params = {}
for col in columns_to_fit:
    print(f"\nProcessing {col}...")
    data = df[col].to_numpy()
    background, adjustment_type, params = estimate_background(data)
    initial_backgrounds[col] = background
    adjustment_types[col] = adjustment_type
    background_params[col] = {'background': background, 'adjustment_type': adjustment_type, **params}
    if adjustment_type == 'exponential':
        # Subtract the background (F(0)) to set cycle 0 to 0.0
        df_adjusted[col] = df[col] - background
    else:  # linear_shift or linear_tilt
        all_cycles = np.arange(1, len(data) + 1)
        if adjustment_type == 'linear_shift':
            # Subtract the intercept to shift to 0.0 at cycle 0
            df_adjusted[col] = df[col] - params['intercept']
        else:  # linear_tilt
            # Apply tilt correction and adjust to 0.0 at cycle 0
            tilt_correction = params['slope'] * all_cycles + params['intercept']
            df_adjusted[col] = df[col] - tilt_correction
            early_adjusted = df_adjusted[col].iloc[1:1 + params['early_cycles']]  # Start at cycle 2
            avg_early = np.mean(early_adjusted)
            df_adjusted[col] -= avg_early  # Normalize early region to minimize offset

print("Background correction method: General exponential fit F(c) = A * x^c + B, extrapolated to cycle 0 (dynamic linear fallback on failure)")
print(f"Linear fallback mode: {'shift+tilt (level baseline)' if APPLY_BASELINE_TILT else 'shift only'}")
print("All data adjusted to extrapolate to 0.0 at cycle 0.")
print("Estimated background values and parameters (using cycles 2-6+, extended dynamically):")
for col in columns_to_fit:
    background = background_params[col]['background']
    adj_type = background_params[col]['adjustment_type']
    if adj_type == 'exponential':
        A = background_params[col]['A']
        x = background_params[col]['x']
        B = background_params[col]['B']
        cycles_used = background_params[col]['cycles']
        param_str = f"A = {A:.6f}, Efficiency (x) = {x:.4f}, B = {B:.4f}, Cycles = {cycles_used}"
    else:
        slope = background_params[col]['slope']
        intercept = background_params[col]['intercept']
        cycles_used = background_params[col]['cycles']
        param_str = f"Slope = {slope:.4f}, Intercept = {intercept:.4f}, Cycles = {cycles_used}"
    print(f"{col}: Background = {background:.4f}, Type = {adj_type}, {param_str}")

if debug_flag:
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    input_base_name = os.path.basename(file_path).replace('.csv', '')
    log_path = os.path.join(output_dir, f'background_parameters_{input_base_name}_{timestamp}.txt')
    with open(log_path, 'w') as f:
        f.write("Background correction method: General exponential fit F(c) = A * x^c + B, extrapolated to cycle 0 (dynamic linear fallback on failure)\n")
        f.write("All data adjusted to extrapolate to 0.0 at cycle 0.\n")
        f.write("Estimated background values and parameters (using cycles 2-6+, extended dynamically):\n")
        for col in columns_to_fit:
            background = background_params[col]['background']
            adj_type = background_params[col]['adjustment_type']
            if adj_type == 'exponential':
                A = background_params[col]['A']
                x = background_params[col]['x']
                B = background_params[col]['B']
                cycles_used = background_params[col]['cycles']
                param_str = f"A = {A:.6f}, Efficiency (x) = {x:.4f}, B = {B:.4f}, Cycles = {cycles_used}"
            else:
                slope = background_params[col]['slope']
                intercept = background_params[col]['intercept']
                cycles_used = background_params[col]['cycles']
                param_str = f"Slope = {slope:.4f}, Intercept = {intercept:.4f}, Cycles = {cycles_used}"
            f.write(f"{col}: Background = {background:.4f}, Type = {adj_type}, {param_str}\n")
    print(f"Saved (Debug) background parameters to {log_path}")

print("\nUnadjusted and Adjusted Fluorescence Values for Cycle 1:")
report_data = {
    'Sample': [],
    'Unadjusted_Cycle1': [],
    'Adjusted_Cycle1': []
}
for col in columns_to_fit:
    unadjusted = df[col].iloc[0]
    adjusted = df_adjusted[col].iloc[0]
    report_data['Sample'].append(col)
    report_data['Unadjusted_Cycle1'].append(unadjusted)
    report_data['Adjusted_Cycle1'].append(adjusted)
report_df = pd.DataFrame(report_data)
pd.set_option('display.float_format', '{:.4f}'.format)
print(report_df.to_string(index=False))

if debug_flag:
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    input_base_name = os.path.basename(file_path).replace('.csv', '')
    report_path = os.path.join(output_dir, f'cycle_1_comparison_{input_base_name}_{timestamp}.csv')
    report_df.to_csv(report_path, index=False)
    print(f"Saved (Debug) cycle 1 comparison to {report_path}")

    output_path = os.path.join(output_dir, f'adjusted_qpcr_data_{input_base_name}_{timestamp}.csv')
    df_adjusted.to_csv(output_path, index=False)
    print(f"Saved (Debug) adjusted data to {output_path}")

def plot_original_vs_adjusted(df_original, df_adjusted, columns, num_cycles=10, debug_flag=False, debug_display_flag=False, output_dir="outputs", file_path=""):
    if not columns:
        print("No columns to plot.")
        return

    if not debug_display_flag and not debug_flag:
        print("Debug output disabled. Skipping plot generation.")
        return

    plt.figure(figsize=(12, 8), constrained_layout=True)
    cycles = np.arange(1, min(num_cycles + 1, len(df_original) + 1))
    colors = plt.cm.tab10(np.linspace(0, 1, len(columns)))
    for idx, column in enumerate(columns):
        raw_data = df_original[column].iloc[:num_cycles].to_numpy()
        adjusted_data = df_adjusted[column].iloc[:num_cycles].to_numpy()
        plt.plot(cycles, raw_data, label=f'{column} Original', marker='o', linestyle='-', color=colors[idx])
        plt.plot(cycles, adjusted_data, label=f'{column} Adjusted', marker='x', linestyle='--', color=colors[idx])
    plt.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    plt.xlabel('Cycle')
    plt.ylabel('Fluorescence Value')
    plt.title(f'Original vs Background-Subtracted Fluorescence: First {num_cycles} Cycles (Raw)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2 if len(columns) > 10 else 1)
    plt.grid(True)

    if debug_flag:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        input_base_name = os.path.basename(file_path).replace('.csv', '')
        plot_path = os.path.join(output_dir, f'background_adjusted_plot_first_10_{input_base_name}_{timestamp}.png')
        plt.savefig(plot_path, bbox_inches='tight')
        print(f"Saved (Debug) first 10 cycles plot to {plot_path}")

    if debug_display_flag:
        plt.show()
        print("Plot displayed successfully.")
    else:
        plt.close()

def plot_adjusted_all_cycles(df_adjusted, columns, debug_flag=False, debug_display_flag=False, output_dir="outputs", file_path=""):
    if not columns:
        print("No columns to plot.")
        return

    if not debug_display_flag and not debug_flag:
        print("Debug output disabled. Skipping plot generation.")
        return

    plt.figure(figsize=(12, 8), constrained_layout=True)
    cycles = np.arange(1, len(df_adjusted) + 1)
    colors = plt.cm.tab10(np.linspace(0, 1, len(columns)))
    for idx, column in enumerate(columns):
        adjusted_data = df_adjusted[column].to_numpy()
        plt.plot(cycles, adjusted_data, label=f'{column}', marker='o', linestyle='-', color=colors[idx])
    plt.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    plt.xlabel('Cycle')
    plt.ylabel('Fluorescence Value')
    plt.title(f'Background-Subtracted Fluorescence: All Cycles (Raw)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2 if len(columns) > 10 else 1)
    plt.grid(True)

    if debug_flag:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        input_base_name = os.path.basename(file_path).replace('.csv', '')
        plot_path = os.path.join(output_dir, f'background_adjusted_plot_all_cycles_{input_base_name}_{timestamp}.png')
        plt.savefig(plot_path, bbox_inches='tight')
        print(f"Saved (Debug) all cycles plot to {plot_path}")

    if debug_display_flag:
        plt.show()
        print("Plot displayed successfully.")
    else:
        plt.close()

if columns_to_fit:
    plot_original_vs_adjusted(
        df,
        df_adjusted,
        columns_to_fit,
        debug_flag=debug_flag,
        debug_display_flag=debug_display_flag,
        output_dir=output_dir,
        file_path=file_path
    )
    plot_adjusted_all_cycles(
        df_adjusted,
        columns_to_fit,
        debug_flag=debug_flag,
        debug_display_flag=debug_display_flag,
        output_dir=output_dir,
        file_path=file_path
    )
else:
    print("No columns to plot.")

# ============================================================================
# SNR-Based Amplification Detection
# ============================================================================
# After baseline subtraction, check if each sample has substantial signal increase.
# Samples with low signal-to-noise ratio are flagged early to save computation.
# ============================================================================

MIN_SIGNAL_TO_NOISE_RATIO = 25  # Configurable threshold; increase to reduce false positives on drifting baselines
NOISE_WINDOW_CYCLES = 5  # Number of early cycles to estimate baseline noise

print("\n" + "="*60)
print("Signal-to-Noise Ratio Analysis (SNR)")
print("="*60)
print(f"Threshold: SNR >= {MIN_SIGNAL_TO_NOISE_RATIO} required for substantial amplification")
print(f"Noise estimated from first {NOISE_WINDOW_CYCLES} cycles of baseline-subtracted data\n")

amplification_flags = {}
snr_details = {}

for col in columns_to_fit:
    adjusted_data = df_adjusted[col].to_numpy()

    # Calculate noise from early cycles (should be near 0 after baseline subtraction)
    noise_cycles = min(NOISE_WINDOW_CYCLES, len(adjusted_data))
    early_data = adjusted_data[:noise_cycles]
    noise = np.std(early_data)

    # Calculate max signal
    max_signal = np.max(adjusted_data)

    # Calculate SNR (protect against division by zero)
    if noise > 0:
        snr = max_signal / noise
    else:
        # If noise is 0 and max_signal > 0, that's infinite SNR (good)
        # If both are 0, no signal at all
        snr = np.inf if max_signal > 0 else 0

    # Store details
    snr_details[col] = {'noise': noise, 'max_signal': max_signal, 'snr': snr}

    # Flag samples with low SNR
    if snr < MIN_SIGNAL_TO_NOISE_RATIO:
        amplification_flags[col] = False
        print(f"⚠️  {col}: No substantial amplification (SNR={snr:.1f}, max={max_signal:.1f}, noise={noise:.1f})")
    else:
        amplification_flags[col] = True
        print(f"✓  {col}: Substantial amplification detected (SNR={snr:.1f})")

# Summary
amplified_count = sum(1 for v in amplification_flags.values() if v)
flagged_count = len(amplification_flags) - amplified_count
print(f"\n=== SNR Amplification Summary ===")
print(f"Samples with substantial amplification: {amplified_count}")
print(f"Samples flagged (no substantial amplification): {flagged_count}")
if flagged_count > 0:
    flagged_samples = [col for col, flag in amplification_flags.items() if not flag]
    print(f"Flagged samples: {', '.join(flagged_samples)}")
print("="*60)

# Outputs: df_adjusted, initial_backgrounds, adjustment_types, amplification_flags


In [ ]:
#==================  Cq and GF Combined - Cell-4  ======================
# Cell-4: Compute shifted fluorescence data (current relative to previous) for initial global fitting to refine baselines
"""
Purpose:
  - Define a function to shift fluorescence data into 'prev' (cycle n-1) and 'current' (cycle n) values.
  - Compute shifted data for each column using baseline-adjusted fluorescence data (df_adjusted).
  - Preview the first 5 and last 5 cycles of shifted data (prev and current) for all columns in a table.
  - Generate detailed reports and save files in Debug mode.

Inputs:
- df_adjusted: DataFrame with baseline-adjusted fluorescence data (background-subtracted from Cell-3).
- columns_to_fit: List of column names to process (from Cell-3).
- debug_flag: True or False, indicating Debug mode (from Cell-1).
- eval_flag: True or False, indicating Evaluation mode (from Cell-1).
- output_dir: String, path to the outputs directory (from Cell-1).
- file_path: String, path to the qPCR data file (from Cell-1).

Outputs:
- shifted_data: Dictionary mapping each column to its shifted data (prev and current fluorescence values).
- (Optional, Debug): Saved shifted data (CSV) in outputs directory.
"""

import pandas as pd
import numpy as np
import datetime
import os

# Verify inputs from Cell-3
try:
    df_adjusted, columns_to_fit
except NameError:
    raise NameError("Required variables (df_adjusted, columns_to_fit) not defined. Please run Cell-3 first.")

# Verify amplification flags from Cell-3 (SNR-based detection)
try:
    amplification_flags
except NameError:
    amplification_flags = {}
    print("Note: amplification_flags not found; all samples will be processed.")

# Verify inputs from Cell-1
try:
    debug_flag, eval_flag, output_dir, file_path
except NameError:
    raise NameError("Required variables (debug_flag, eval_flag, output_dir, file_path) not defined. Please run Cell-1 first.")

# Initialize output
shifted_data = {}

# Step 1: Define function to shift data
def shift_data(df, columns):
    """
    Shift fluorescence data into prev (cycle n-1) and current (cycle n) values.

    Args:
        df (pd.DataFrame): DataFrame with fluorescence data.
        columns (list): List of column names to process.

    Returns:
        dict: Dictionary mapping each column to {'prev': array, 'current': array}.
    """
    shifted_data = {}
    for col in columns:
        prev = df[col].iloc[:-1].to_numpy()
        current = df[col].iloc[1:].to_numpy()
        shifted_data[col] = {'prev': prev, 'current': current}
    return shifted_data

# Step 2: Compute shifted data (only for samples that passed Cell-3)
valid_columns = [col for col in columns_to_fit if amplification_flags.get(col, True)]
shifted_data = shift_data(df_adjusted, valid_columns)

# Step 3: Preview shifted data in a table (first 5 and last 5 cycles)
if valid_columns:
    # Convert shifted data to DataFrame with cycle numbers starting at 2
    cycles = np.arange(2, len(shifted_data[valid_columns[0]]['prev']) + 2)  # 2 to 40 for 39 rows
    shifted_df = pd.DataFrame(index=cycles)
    for col in valid_columns:
        shifted_df[f'{col}_Prev'] = shifted_data[col]['prev']
        shifted_df[f'{col}_Current'] = shifted_data[col]['current']

    print("\nShifted Data Preview (First 5 Cycles):")
    print(shifted_df.head(5).to_string())
    print("\nShifted Data Preview (Last 5 Cycles):")
    print(shifted_df.tail(5).to_string())
else:
    print("No columns to process for shifted data (all samples flagged in Cell-3).")

# Step 4: Debug output: Save shifted data to CSV
if debug_flag and valid_columns:
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    input_base_name = os.path.basename(file_path).replace('.csv', '')
    csv_filename = f"shifted_data_{input_base_name}_{timestamp}.csv"
    csv_path = os.path.join(output_dir, csv_filename)

    max_length = max(len(shifted_data[col]['prev']) for col in valid_columns) if valid_columns else 0
    shifted_df_data = {'Cycle': np.arange(2, max_length + 2)}  # Cycles start at 2
    for col in valid_columns:
        prev = shifted_data[col]['prev']
        current = shifted_data[col]['current']
        prev_padded = np.pad(prev, (0, max_length - len(prev)), constant_values=np.nan)
        current_padded = np.pad(current, (0, max_length - len(current)), constant_values=np.nan)
        shifted_df_data[f"{col}_Prev"] = prev_padded
        shifted_df_data[f"{col}_Current"] = current_padded

    shifted_df = pd.DataFrame(shifted_df_data)
    with open(csv_path, 'w') as f:
        f.write(f"# Results from input file: {file_path}\n")
        f.write(f"# Generated on: {timestamp}\n")
        shifted_df.to_csv(f, index=False)
    print(f"Saved (Debug) shifted data to {csv_path}")

# Output: shifted_data


In [ ]:
#==================  Cq and GF Combined - Cell-5  ======================
# Cell-5: Baseline re-adjustment using PCR model (optional)
"""
Purpose:
- Optionally re-adjust baseline using the PCR model to improve agreement between modeled and observed data.
- Fit a recursive PCR model to the shifted fluorescence data and re-adjust the baseline by minimizing SOS between experimental and modeled data.
- For exponential-adjusted samples, can toggle "ALLOW_EXPONENTIAL_FINE_TUNING = False" to return the original shifted data as the best-positioned data.
- Output a DataFrame with best-positioned adjusted data for all samples.
- Plot observed vs. initial (red dashed) and final (blue dashed) modeled fluorescence data for each sample.
- Export basic results for inspection.

Inputs:
- shifted_data: Dictionary with shifted fluorescence data (prev and current values) from Cell-4.
- columns_to_fit: List of column names to process from Cell-2.
- adjustment_types: Dictionary mapping each column to adjustment type from Cell-3 ('exponential', 'linear_shift', or 'linear_tilt').
- file_path: String, path to the qPCR data file from Cell-1.
- debug_flag: True or False, indicating Debug mode (from Cell-1).
- debug_display_flag: True or False, indicating Debug plot display (from Cell-1).
- eval_flag: True or False, indicating Evaluation mode (from Cell-1).
- output_dir: String, path to the outputs directory from Cell-1.

Outputs:
- df_fine_tuned: DataFrame with fine-tuned fluorescence data for all samples (refined for linear, original for exponential).
- fitted_params: Dictionary with initial and final fitted parameters (max_val, KD, max_val/KD) and SOS for each sample.
- model_predictions: Dictionary with initial and final model predictions for each sample.
- Plots: Observed vs. initial and final fits for each sample (Debug display only).
- Basic text output with parameters and SOS.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
from scipy.optimize import leastsq, minimize_scalar
import os

# Verify inputs
try:
    shifted_data, columns_to_fit, df, adjustment_types, file_path
except NameError:
    raise NameError("Required variables (shifted_data, columns_to_fit, df, adjustment_types, file_path) not defined. Please run Cell-2, Cell-3, and Cell-4 first.")

# Verify inputs from Cell-1
try:
    debug_flag, debug_display_flag, eval_flag, output_dir
except NameError:
    raise NameError("Required variables (debug_flag, debug_display_flag, eval_flag, output_dir) not defined. Please run Cell-1 first.")

# Use amplification flags from Cell-3 when available (skip plots/reports for flagged samples)
try:
    amplification_flags
except NameError:
    amplification_flags = {}

# Evaluation/debug flags (Cell-1)
try:
    eval_flag
except NameError:
    eval_flag = False

try:
    debug_flag
except NameError:
    debug_flag = False

try:
    debug_display_flag
except NameError:
    debug_display_flag = False

# Initialize outputs
adjusted_df = pd.DataFrame(index=np.arange(1, len(next(iter(shifted_data.values()))['current']) + 2))  # Initialize with cycles 1-40
df_fine_tuned = df_adjusted.copy()  # Start with baseline-adjusted data from Cell 3
fitted_params = {}
model_predictions = {}

# Fitter class for recursive PCR model
class Fitter:
    def __init__(self, prev, current, initial_max_observed):
        self.prev = prev
        self.current = current
        self.initial_max_observed = initial_max_observed

    def pcr_model(self, prev, max_val, KD):
        """Recursive PCR model with non-negative constraint"""
        return np.maximum(0, prev * (1 + ((max_val - prev) / max_val) - (prev / (KD + prev))))

    def residuals(self, params, prev_subset, current_subset):
        """Compute residuals for least-squares optimization"""
        max_val, KD = params
        y_pred = self.pcr_model(prev_subset, max_val, KD)
        return y_pred - current_subset

    def fit_leastsq(self, max_val_guess, KD_guess, start_idx=0):
        """Fit the model using least squares from start_idx onward"""
        if len(self.prev[start_idx:]) < 2:
            print(f"Error: Insufficient data points (less than 2) from index {start_idx} for fitting.")
            return max_val_guess, KD_guess, np.array([])

        prev_subset = self.prev[start_idx:]
        current_subset = self.current[start_idx:]

        p0 = [max_val_guess, KD_guess]
        print(f"Leastsq initial guesses: max_val={p0[0]:.4f}, KD={p0[1]:.4f}, "
              f"Fitting from cycle {start_idx + 2}")

        try:
            params, _ = leastsq(self.residuals, p0, args=(prev_subset, current_subset), maxfev=10000)
            if params[0] <= 0 or params[1] <= 0:
                print(f"Warning: Non-positive parameters detected (max_val={params[0]:.4f}, KD={params[1]:.4f}). Returning guesses.")
                return max_val_guess, KD_guess, np.array([])
            print(f"Fitted parameters: max_val={params[0]:.4f}, KD={params[1]:.4f}")
            y_pred = self.pcr_model(prev_subset, params[0], params[1])
            residuals = y_pred - current_subset
            return params[0], params[1], residuals
        except Exception as e:
            print(f"Leastsq failed: {e}. Returning initial guesses.")
            return max_val_guess, KD_guess, np.array([])

    def fit_max_val_KD(self):
        """Estimate initial guesses and fit the model using fixed initial max_observed"""
        max_val_guess = self.initial_max_observed * 5
        KD_guess = self.initial_max_observed * 0.2
        return self.fit_leastsq(max_val_guess, KD_guess)

# Toggle: allow fine-tuning of baseline-adjusted data (default: False)
ALLOW_FINE_TUNING = False

# Step 1: Initial fit with original shifted data
for col in columns_to_fit:
    print(f"\nInitial fitting PCR model for {col}")
    if col not in shifted_data:
        print(f"Warning: No shifted data for {col}. Skipping.")
        continue
    initial_max_observed = max(shifted_data[col]['current'])
    fitter = Fitter(shifted_data[col]['prev'], shifted_data[col]['current'], initial_max_observed)
    max_val_init, KD_init, residuals_init = fitter.fit_max_val_KD()

    # Compute max_val/KD ratio
    max_val_over_KD_init = max_val_init / KD_init if KD_init != 0 else "Undefined (KD=0)"
    initial_sos = np.sum(residuals_init ** 2) if len(residuals_init) > 0 else float('inf')

    # Store initial parameters
    fitted_params[col] = {
        'max_val': max_val_init,
        'KD': KD_init,
        'max_val_over_KD': max_val_over_KD_init,
        'initial_sos': initial_sos
    }
    model_predictions[col] = {'initial': fitter.pcr_model(shifted_data[col]['prev'], max_val_init, KD_init)}
    # Populate adjusted_df with initial shifted data (will be overwritten for linear samples)
    adjusted_df[col] = np.pad(shifted_data[col]['current'], (1, 0), mode='constant', constant_values=np.nan)  # Pad with NaN at cycle 1

# Step 2: Conditional optimization based on adjustment_types
if not ALLOW_FINE_TUNING:
    print("\nFine-tuning disabled: using baseline-adjusted data from Cell-3 without additional optimization.")
    for col in columns_to_fit:
        if col not in fitted_params:
            continue
        fitted_params[col].update({
            'final_max_val': fitted_params[col]['max_val'],
            'final_KD': fitted_params[col]['KD'],
            'final_max_val_over_KD': fitted_params[col]['max_val_over_KD'],
            'final_sos': fitted_params[col]['initial_sos'],
            'baseline_adjustment': 0.0
        })
        if col in model_predictions:
            model_predictions[col]['final'] = model_predictions[col]['initial']
else:
    # Toggle to allow fine-tuning of exponential-adjusted samples (experimental)
    ALLOW_EXPONENTIAL_FINE_TUNING = True
    for col in columns_to_fit:
        if col not in shifted_data or col not in adjustment_types:
            continue
        adj_type = adjustment_types[col]
        print(f"\nProcessing {col} (adjustment_type: {adj_type})")
        initial_max_observed = max(shifted_data[col]['current'])

        if adj_type == 'exponential' and not ALLOW_EXPONENTIAL_FINE_TUNING:
            # Use original baseline-adjusted data for exponential (no further refinement needed)
            print(f"Using original baseline-adjusted data for exponential-adjusted sample {col}.")
            adjusted_df[col] = np.pad(shifted_data[col]['current'], (1, 0), mode='constant', constant_values=np.nan)  # Pad with NaN at cycle 1
            # df_fine_tuned[col] already contains the baseline-adjusted data from Cell 3, no change needed
            max_val_final = fitted_params[col]['max_val']
            KD_final = fitted_params[col]['KD']
            max_val_over_KD_final = fitted_params[col]['max_val_over_KD']
            final_sos = fitted_params[col]['initial_sos']
            best_adjustment = 0.0  # No adjustment
            model_predictions[col]['final'] = model_predictions[col]['initial']
        else:
            # Proceed with refinement for linear
            print(f"Optimizing baseline adjustment for {col} (adjustment_type: {adj_type})")

            def objective_function(adjustment):
                prev_variant = shifted_data[col]['prev'] + adjustment
                current_variant = shifted_data[col]['current'] + adjustment
                fitter = Fitter(prev_variant, current_variant, initial_max_observed)
                max_val, KD, residuals = fitter.fit_max_val_KD()
                sos = np.sum(residuals ** 2) if len(residuals) > 0 else float('inf')
                print(f"Evaluating adjustment {adjustment:.4f}, SOS = {sos:.4f}")
                return sos

            # Use auto-bracketing with a starting point
            result = minimize_scalar(objective_function, method='brent', options={'maxiter': 50, 'xtol': 1e-4})

            # Apply best adjustment
            best_adjustment = result.x
            prev_variant = shifted_data[col]['prev'] + best_adjustment
            current_variant = shifted_data[col]['current'] + best_adjustment
            # Ensure current_variant has the same length as shifted_data[col]['current']
            if len(current_variant) != len(shifted_data[col]['current']):
                print(f"Warning: Length of current_variant for {col} ({len(current_variant)}) does not match "
                      f"shifted_data['{col}']['current'] ({len(shifted_data[col]['current'])}). Padding with NaN.")
                current_variant = np.pad(current_variant, (0, len(shifted_data[col]['current']) - len(current_variant)),
                                        mode='constant', constant_values=np.nan)
            # Apply adjustment to the full column, including cycle 1
            full_adjusted = df_fine_tuned[col].values + best_adjustment  # Add adjustment to baseline-adjusted data (cycle 1-n)
            df_fine_tuned[col] = full_adjusted  # Update df_fine_tuned with refined data
            fitter_final = Fitter(prev_variant, current_variant, initial_max_observed)
            max_val_final, KD_final, residuals_final = fitter_final.fit_max_val_KD()

            # Compute max_val/KD ratio
            max_val_over_KD_final = max_val_final / KD_final if KD_final != 0 else "Undefined (KD=0)"
            final_sos = np.sum(residuals_final ** 2) if len(residuals_final) > 0 else float('inf')
            model_predictions[col]['final'] = fitter_final.pcr_model(prev_variant, max_val_final, KD_final)

        # Store final parameters
        fitted_params[col].update({
            'final_max_val': max_val_final,
            'final_KD': KD_final,
            'final_max_val_over_KD': max_val_over_KD_final,
            'final_sos': final_sos,
            'baseline_adjustment': best_adjustment
        })

# Step 3: Export final adjusted data (only those that passed Cell-3)
valid_columns = [col for col in columns_to_fit if amplification_flags.get(col, True)]
if (debug_flag or eval_flag) and valid_columns:
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    input_base_name = os.path.basename(file_path).replace('.csv', '')
    export_filename = f"final_adjusted_data_{input_base_name}_{timestamp}.csv"
    export_path = os.path.join(output_dir, export_filename)

    export_df = df_fine_tuned[valid_columns].copy()
    export_df.insert(0, 'Cycle', np.arange(1, len(export_df) + 1))
    export_df.to_csv(export_path, index=False)
    print(f"Saved (Debug/Evaluation) final adjusted data to {export_path}")

# Step 4: Plot for each sample (only those that passed Cell-3)
if debug_display_flag:
    for col in valid_columns:
        if col in model_predictions and col in shifted_data:
            cycles = np.arange(1, len(df_fine_tuned[col]) + 1)  # Use length of df_fine_tuned to ensure cycle 1 is included
            plt.figure(figsize=(12, 8), constrained_layout=True)
            plt.plot(cycles, df_fine_tuned[col].values, label=f'{col} Observed', marker='o', linestyle='None', color='black')
            plt.plot(cycles[1:], model_predictions[col]['initial'], label=f'{col} Initial Model', linestyle='--', color='red')
            plt.plot(cycles[1:], model_predictions[col]['final'], label=f'{col} Final Model', linestyle='--', color='blue')
            plt.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
            plt.xlabel('Cycle')
            plt.ylabel('Fluorescence Value')
            max_val_over_KD_str = f"{fitted_params[col]['final_max_val_over_KD']:.2f}" if isinstance(fitted_params[col]['final_max_val_over_KD'], (int, float)) else fitted_params[col]['final_max_val_over_KD']
            plt.title(f'{col}: Observed (Black) from 1-n vs Initial (Red) and Final (Blue) Model from 2-n\n'
                      f'(Initial: max_val={fitted_params[col]["max_val"]:.2f}, KD={fitted_params[col]["KD"]:.2f})\n'
                      f'(Final: max_val={fitted_params[col]["final_max_val"]:.2f}, KD={fitted_params[col]["final_KD"]:.2f}, max_val/KD={max_val_over_KD_str})')
            plt.legend()
            plt.grid(True)

            if debug_flag:
                timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                input_base_name = os.path.basename(file_path).replace('.csv', '')
                plot_filename = f"pcr_model_fit_{col}_{input_base_name}_{timestamp}.png"
                plot_path = os.path.join(output_dir, plot_filename)
                plt.savefig(plot_path, bbox_inches='tight')
                print(f"Saved (Debug) plot to {plot_path}")

            plt.show()
            print("Plot displayed successfully.")
else:
    print("Debug display disabled. Skipping plot generation.")

# Outputs: df_fine_tuned, fitted_params, model_predictions


In [ ]:
#==================  Cq and GF Combined - Cell-6  ======================
# Cell-6: Log-transform refined fluorescence data for Cq analysis
"""
Purpose:
- Perform log10 transformation on the refined fluorescence data from Cell-5.
- Replace negative or zero values with a small positive value (1E-4) to avoid log(0) errors and improve scaling.
- Preview the log-transformed data in a table.
- Plot the log-transformed data for all samples on a single superimposed plot to evaluate sample-to-sample variations.
- Save the log-transformed data and plot in Debug mode.

Inputs:
- df_fine_tuned: DataFrame with refined fluorescence data for all samples (from Cell-5).
- columns_to_fit: List of column names to process (from Cell-2).
- file_path: String, path to the qPCR data file (from Cell-1).
- debug_flag: True or False, indicating Debug mode (from Cell-1).
- debug_display_flag: True or False, indicating Debug plot display (from Cell-1).
- eval_flag: True or False, indicating Evaluation mode (from Cell-1).
- output_dir: String, path to output directory (from Cell-1).

Outputs:
- df_log_refined: DataFrame with log10-transformed refined fluorescence data.
- (Optional, Debug): Saved log-transformed data (CSV) and superimposed plot (PNG).
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import datetime

# Verify inputs from Cell-5
try:
    df_fine_tuned, columns_to_fit
except NameError:
    raise NameError("Required variables (df_fine_tuned, columns_to_fit) not defined. Please run Cell-5 first.")

# Verify inputs from Cell-1
try:
    file_path, debug_flag, debug_display_flag, eval_flag, output_dir
except NameError:
    raise NameError("file_path, debug_flag, debug_display_flag, eval_flag, or output_dir not defined. Please run Cell-1 first.")

# Use amplification flags from Cell-3 when available (skip plots/reports for flagged samples)
try:
    amplification_flags
except NameError:
    amplification_flags = {}

# Step 1: Log-transform the refined fluorescence data
df_log_refined = df_fine_tuned.copy()
small_value = 1e-4  # Use 1E-4 to pad non-positive values
for col in columns_to_fit:
    # Replace non-positive values with a small positive value before log transformation
    df_log_refined[col] = np.log10(np.where(df_fine_tuned[col] <= 0, small_value, df_fine_tuned[col]))

# Step 2: Preview the log-transformed data (only samples that passed Cell-3)
valid_columns = [col for col in columns_to_fit if amplification_flags.get(col, True)]
print("\nLog-Transformed Refined Fluorescence Data (Non-positive values padded with 1E-4):")
preview_cols = valid_columns if valid_columns else columns_to_fit
preview_data = df_log_refined[preview_cols]
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print(preview_data.to_string(index=True))

# Step 3: Plot log-transformed data for all samples on a single plot
def plot_log_transformed_data(df_log, columns, debug_flag=False, debug_display_flag=False, output_dir="outputs", file_path=""):
    """Plot log-transformed data for all samples"""
    if not debug_display_flag and not debug_flag:
        print("Debug output disabled. Skipping plot generation.")
        return

    cycles = np.arange(1, len(df_log) + 1)
    plt.figure(figsize=(12, 8), constrained_layout=True)
    colors = plt.cm.tab10(np.linspace(0, 1, len(columns)))
    for idx, col in enumerate(columns):
        log_data = df_log[col]
        plt.plot(cycles, log_data, label=f'{col}', marker='o', linestyle='-', color=colors[idx])

    plt.axhline(y=np.log10(small_value), color='gray', linestyle='--', linewidth=0.5, label='Baseline (1E-4)')
    plt.xlabel('Cycle')
    plt.ylabel('Log10(Fluorescence)')
    plt.title('Log-Transformed Refined Data for All Samples')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)

    # Debug output: Save the plot
    if debug_flag:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        input_base_name = os.path.basename(file_path).replace('.csv', '')
        plot_filename = f"log_transformed_all_{input_base_name}_{timestamp}.png"
        plot_path = os.path.join(output_dir, plot_filename)
        plt.savefig(plot_path, bbox_inches='tight')
        print(f"Saved (Debug) plot to {plot_path}")

    if debug_display_flag:
        plt.show()
        print("Plot displayed successfully.")
    else:
        plt.close()

# Plot the log-transformed data (only samples that passed Cell-3)
if valid_columns:
    plot_log_transformed_data(df_log_refined, valid_columns, debug_flag, debug_display_flag, output_dir, file_path)
else:
    print("No columns to plot.")

# Step 4: Debug output: Save log-transformed data to CSV
if debug_flag:
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    input_base_name = os.path.basename(file_path).replace('.csv', '')
    csv_filename = f"log_transformed_data_{input_base_name}_{timestamp}.csv"
    csv_path = os.path.join(output_dir, csv_filename)
    df_log_refined.to_csv(csv_path)
    print(f"Saved (Debug) log-transformed data to {csv_path}")

# Outputs: df_log_refined


In [ ]:
#==================  Cq and GF Combined - Cell-7  ======================
# Cell-7: Identify steepest exponential phase with sliding window for Cq analysis
"""
Purpose:
- Identify the steepest exponential phase in the log-transformed refined fluorescence data (df_log_refined) for each column:
  - Find the first 4-cycle window in df_fine_tuned where fluorescence values are strictly increasing, starting from cycle 2.
  - Start a sliding window from that point and fit linear regressions on log-transformed data (df_log_refined).
- Select the top 2 windows with the highest R^2 (best linear fits).
- Choose the window with the highest slope among those two.
- Slope constraints prevent flat/near‑flat regions from being selected.
- Samples already flagged in Cell 3 (SNR-based detection) are skipped.
- For samples with no identifiable exponential phase, store "No exponential phase identified for sample N" as output.
- For samples where a window is found but the slope indicates no substantial amplification (slope < 0.15,
  corresponding to <41% PCR efficiency), flag as "No substantial amplification" and exclude from subsequent analysis.
- Preview the selected window details (start cycle, slope, intercept, R^2) or the no-phase message.
- Plot the log-transformed data (positive values only) with the selected window's regression overlaid as a dashed red line.
- Generate detailed reports and save files in Debug/Evaluation mode.

Inputs:
- df_fine_tuned: DataFrame with refined fluorescence data (from Cell-5).
- df_log_refined: DataFrame with log10-transformed refined fluorescence data (from Cell-6).
- columns_to_fit: List of column names to process (from Cell-2).
- amplification_flags: Dictionary from Cell-3 (SNR-based detection), may already have flagged samples.
- file_path: String, path to the qPCR data file (from Cell-1).
- debug_flag: True or False, indicating Debug mode (from Cell-1).
- debug_display_flag: True or False, indicating Debug plot display (from Cell-1).
- eval_flag: True or False, indicating Evaluation mode (from Cell-1).
- output_dir: String, path to output directory (from Cell-1).

Outputs:
- steepest_windows: Dictionary mapping each column to its selected window's details (start_cycle, slope, intercept, r_squared) or "No exponential phase identified for sample N" for non-amplifying samples.
- amplification_flags: Dictionary mapping each column to True (valid for processing) or False (no substantial amplification, skip in subsequent cells). Updated with slope-based detection results.
- (Optional, Debug/Evaluation): Saved window details (CSV). Plot saved/displayed in Debug.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import os
import datetime

# Verify inputs from previous cells
try:
    df_fine_tuned, df_log_refined, columns_to_fit
except NameError:
    raise NameError("Required variables (df_fine_tuned, df_log_refined, columns_to_fit) not defined. Please run Cell-1, Cell-2, Cell-3, Cell-5, and Cell-6 first.")

try:
    file_path, debug_flag, debug_display_flag, eval_flag, output_dir
except NameError:
    raise NameError("file_path, debug_flag, debug_display_flag, eval_flag, or output_dir not defined. Please run Cell-1 first.")

# Evaluation/debug flags (Cell-1)
try:
    eval_flag
except NameError:
    eval_flag = False

try:
    debug_flag
except NameError:
    debug_flag = False

try:
    debug_display_flag
except NameError:
    debug_display_flag = False

# Check for amplification_flags from Cell-3 (SNR-based detection)
try:
    amplification_flags
    print(f"Using amplification flags from Cell-3 (SNR-based detection)")
    pre_flagged = [col for col, flag in amplification_flags.items() if not flag]
    if pre_flagged:
        print(f"  Already flagged (no substantial amplification): {', '.join(pre_flagged)}")
except NameError:
    amplification_flags = {}
    print("Note: No amplification_flags from Cell-3. Will create fresh flags.")

# Initialize outputs
steepest_windows = {}

# Parameters
window_size = 4  # Size of the sliding window (empirically optimal: 3 points too noisy, longer windows unrealistic)
min_slope = 0.04  # Minimum slope to avoid flat regions (~1.1-fold amplification per cycle)
max_slope = 0.4  # Maximum slope to allow room for higher efficiencies (~2.5-fold amplification per cycle)
# Minimum slope to qualify as "real" PCR amplification (not just noise/drift)
# 0.15 in log10 space = 10^0.15 = 1.41x per cycle = ~41% PCR efficiency
# Real PCR typically shows >60-80% efficiency (slopes >0.20-0.26)
MIN_REAL_AMPLIFICATION_SLOPE = 0.15
num_cycles = len(df_log_refined)

print("\nIdentifying exponential phase windows for Cq analysis...")
print(f"Parameters: window_size={window_size}, min_slope={min_slope}, max_slope={max_slope}")

# Step 1: Find the best window for each sample
for col in columns_to_fit:
    # Skip samples already flagged by Cell-3 (SNR-based detection)
    if col in amplification_flags and not amplification_flags[col]:
        steepest_windows[col] = f"Skipped (flagged in Cell-3: no substantial amplification)"
        print(f"⏭️  {col}: Skipped (already flagged in Cell-3)")
        continue
    log_data = df_log_refined[col].to_numpy()
    refined_data = df_fine_tuned[col].to_numpy()
    window_fits = []

    # Find the first 4-cycle window where refined fluorescence values are strictly increasing, starting from cycle 2 (index 1)
    start_cycle = None
    for start in range(1, num_cycles - window_size + 1):  # Start from index 1 (cycle 2)
        window_refined_data = refined_data[start:start + window_size]
        if (len(window_refined_data) == window_size and
            np.all(np.diff(window_refined_data) > 0)):  # All differences must be positive (strictly increasing)
            start_cycle = start
            break

    if start_cycle is None:
        steepest_windows[col] = f"No exponential phase identified for sample {col}"
        amplification_flags[col] = False  # Flag for skipping in subsequent cells
        print(f"Warning: {steepest_windows[col]}")
        continue

    # Step 2: Slide the window from start_cycle and fit regressions
    for start in range(start_cycle, num_cycles - window_size + 1):
        window_cycles = np.arange(start + 1, start + window_size + 1)  # 1-based cycles
        window_log_data = log_data[start:start + window_size]

        # Skip if window has NaN or insufficient data
        if (len(window_log_data) != window_size or
            np.any(np.isnan(window_log_data))):
            continue

        # Fit linear regression on log-transformed data
        slope, intercept, r_value, _, _ = linregress(window_cycles, window_log_data)
        r_squared = r_value ** 2

        # Store fit details if slope is within the boundaries
        if min_slope < slope <= max_slope:
            window_fits.append({
                'start_cycle': start + 1,
                'slope': slope,
                'intercept': intercept,
                'r_squared': r_squared
            })

    # Step 3: Select the top 2 windows by R², then pick the one with the highest slope
    if len(window_fits) < 2:
        print(f"Warning: Fewer than 2 valid windows found for {col} after starting at cycle {start_cycle + 1}. Skipping.")
        steepest_windows[col] = f"No exponential phase identified for sample {col}"
        amplification_flags[col] = False  # Flag for skipping in subsequent cells
        continue

    # Sort by R² (highest first), take top 2, then select by highest slope
    window_fits.sort(key=lambda x: x['r_squared'], reverse=True)
    top_two = window_fits[:2]
    best_fit = max(top_two, key=lambda x: x['slope'])

    # Step 4: Check if the best window represents real PCR amplification
    # Slope < 0.15 indicates <41% efficiency, which is noise/drift, not biochemically relevant amplification
    if best_fit['slope'] < MIN_REAL_AMPLIFICATION_SLOPE:
        efficiency_pct = (10 ** best_fit['slope'] - 1) * 100
        steepest_windows[col] = (f"No substantial amplification for sample {col} "
                                  f"(slope={best_fit['slope']:.4f}, ~{efficiency_pct:.0f}% efficiency)")
        amplification_flags[col] = False  # Flag for skipping in subsequent cells
        print(f"⚠️  {col}: {steepest_windows[col]}")
        continue

    # Valid amplification detected
    steepest_windows[col] = best_fit
    amplification_flags[col] = True  # Valid for processing in subsequent cells

# Step 4: Preview the selected window details (only samples that passed Cell-3)
valid_columns = [col for col in columns_to_fit if amplification_flags.get(col, True)]
if steepest_windows and valid_columns:
    print("\nSelected Window Details (Top 2 Highest R², Then Highest Slope After First Strictly Increasing 4-Cycle Window):")
    report_data = {
        'Sample': [],
        'Start_Cycle': [],
        'Slope': [],
        'Intercept': [],
        'R_Squared': []
    }
    for col in valid_columns:
        info = steepest_windows.get(col)
        if info is None or isinstance(info, str):  # Handle no-phase case
            continue
        report_data['Sample'].append(col)
        report_data['Start_Cycle'].append(info['start_cycle'])
        report_data['Slope'].append(f"{info['slope']:.4f}")
        report_data['Intercept'].append(f"{info['intercept']:.4f}")
        report_data['R_Squared'].append(f"{info['r_squared']:.4f}")

    report_df = pd.DataFrame(report_data)
    pd.set_option('display.float_format', '{:.4f}'.format)
    if not report_df.empty:
        print(report_df.to_string(index=False))
    else:
        print("No valid exponential windows for samples that passed Cell-3.")
else:
    print("No steepest windows or phase identification for any samples.")

    # Debug/Evaluation output: Save window details as CSV
if (debug_flag or eval_flag) and 'report_df' in locals() and not report_df.empty:
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    input_base_name = os.path.basename(file_path).replace('.csv', '')
    csv_path = os.path.join(output_dir, f'steepest_windows_{input_base_name}_{timestamp}.csv')
    report_df.to_csv(csv_path, index=False)
    print(f"Saved (Debug/Evaluation) steepest window details to {csv_path}")

# Step 5: Plot log-transformed data with selected window regression overlaid
def plot_exponential_windows(df_log, columns, steepest_windows, window_size, debug_flag=False, debug_display_flag=False, output_dir="outputs", file_path=""):
    """Plot log-transformed data with selected exponential windows overlaid"""
    if not debug_display_flag and not debug_flag:
        print("Debug output disabled. Skipping plot generation.")
        return

    cycles = np.arange(1, len(df_log) + 1)
    fig, ax = plt.subplots(figsize=(12, 8))
    colors = plt.cm.tab10(np.linspace(0, 1, len(columns)))

    for idx, col in enumerate(columns):
        log_data = df_log[col].values
        ax.plot(cycles, log_data, label=f'{col}', marker='o', linestyle='-', color=colors[idx])

        if isinstance(steepest_windows[col], dict):
            start = steepest_windows[col]['start_cycle'] - 1  # Convert to 0-based index
            window_cycles = np.arange(start + 1, start + window_size + 1)
            window_data = log_data[start:start + window_size]
            if len(window_cycles) == window_size and not np.all(np.isnan(window_data)):
                regression_line = steepest_windows[col]['slope'] * window_cycles + steepest_windows[col]['intercept']
                ax.plot(window_cycles, regression_line,
                        label=f'{col} Best Window (R²={steepest_windows[col]["r_squared"]:.4f})',
                        linestyle='--', color='red', linewidth=2)

    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_xlabel('Cycle')
    ax.set_ylabel('Log10(Fluorescence)')
    ax.set_title(f'Log10-Transformed Refined Data with Best {window_size}-Cycle Exponential Windows')
    ax.grid(True)

    # Place legend outside without squishing the plot area
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.subplots_adjust(right=0.75)

    # Debug output: Save the plot
    if debug_flag:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        input_base_name = os.path.basename(file_path).replace('.csv', '')
        plot_path = os.path.join(output_dir, f'log_with_best_window_{input_base_name}_{timestamp}.png')
        plt.savefig(plot_path, bbox_inches='tight')
        print(f"Saved (Debug) plot to {plot_path}")

    if debug_display_flag:
        plt.show()
        print("Plot displayed successfully.")
    else:
        plt.close()

# Plot the exponential windows (only samples that passed Cell-3)
valid_columns = [col for col in columns_to_fit if amplification_flags.get(col, True)]
if valid_columns:
    plot_exponential_windows(df_log_refined, valid_columns, steepest_windows, window_size, debug_flag, debug_display_flag, output_dir, file_path)
else:
    print("No columns to plot (all samples flagged in Cell-3).")

# Summary of amplification flags
valid_samples = [col for col, flag in amplification_flags.items() if flag]
invalid_samples = [col for col, flag in amplification_flags.items() if not flag]
print(f"\n=== Amplification Summary (After Cell-7) ===")
print(f"Samples with substantial amplification: {len(valid_samples)} ({', '.join(valid_samples) if valid_samples else 'None'})")
print(f"Samples flagged (no substantial amplification): {len(invalid_samples)} ({', '.join(invalid_samples) if invalid_samples else 'None'})")
if invalid_samples:
    print(f"⚠️  Flagged samples will be excluded from threshold calculation and Cq determination in subsequent cells.")

# Outputs: steepest_windows, amplification_flags


In [ ]:
#==================  Cq and GF Combined - Cell-8  ======================
# Cell-8: Calculate threshold and compute Cq values
"""
Purpose:
- Calculate the threshold as the median of the log fluorescence values at the midpoint of each sample's steepest window's regression line.
- Compute Cq values for each sample using the regression line method:
  Cq is where the 4-point best-fit regression line from Cell-7 crosses the threshold in log space.
- Skip Cq calculation for samples with no exponential phase or no substantial amplification.
- Preview the Cq values in a table.
- Plot the log-transformed data with the threshold and steepest windows' regression lines overlaid.
- Generate detailed reports in Debug/Evaluation mode.
- Export Cq values to a separate CSV file in Debug mode only (comprehensive output is in Cell 11).

Inputs:
- df_fine_tuned: DataFrame with refined fluorescence data (from Cell-5).
- df_log_refined: DataFrame with log10-transformed refined fluorescence data (from Cell-6).
- columns_to_fit: List of column names to process (from Cell-2).
- steepest_windows: Dictionary with steepest window details per column (from Cell-7).
- amplification_flags: Dictionary indicating if each sample showed substantial amplification (from Cell-3/Cell-7).
- file_path: String, path to the qPCR data file (from Cell-1).
- debug_flag: True or False, indicating Debug mode (from Cell-1).
- debug_display_flag: True or False, indicating Debug plot display (from Cell-1).
- eval_flag: True or False, indicating Evaluation mode (from Cell-1).
- output_dir: String, path to output directory (from Cell-1).

Outputs:
- threshold_log: The median log fluorescence threshold.
- threshold_linear: The threshold in linear scale (for reference).
- cq_values: Dictionary mapping each column to its Cq value (regression line crossing point).
- (Optional, Debug): CSV file 'cq_values_[input_base_name]_[timestamp].csv' and saved plot (PNG). Plot displays only in Debug.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import os

# Verify inputs from Cell-5
try:
    df_fine_tuned, columns_to_fit
except NameError:
    raise NameError("Required variables (df_fine_tuned, columns_to_fit) not defined. Please run Cell-5 first.")

# Verify inputs from Cell-6
try:
    df_log_refined
except NameError:
    raise NameError("Required variable (df_log_refined) not defined. Please run Cell-6 first.")

# Verify inputs from Cell-7
try:
    steepest_windows
except NameError:
    raise NameError("Required variable (steepest_windows) not defined. Please run Cell-7 first.")

try:
    amplification_flags
except NameError:
    raise NameError("Required variable (amplification_flags) not defined. Please run Cell-7 first.")

# Verify inputs from Cell-1
try:
    file_path, debug_flag, debug_display_flag, eval_flag, output_dir
except NameError:
    raise NameError("file_path, debug_flag, debug_display_flag, eval_flag, or output_dir not defined. Please run Cell-1 first.")

# Evaluation/debug flags (Cell-1)
try:
    eval_flag
except NameError:
    eval_flag = False

try:
    debug_flag
except NameError:
    debug_flag = False

try:
    debug_display_flag
except NameError:
    debug_display_flag = False

# Initialize outputs
threshold_log = None
threshold_linear = None
cq_values = {}

# Parameters
window_size = 4  # Matches Cell-7's window_size
num_cycles = len(df_fine_tuned)

print("Calculating Cq values...")

# Step 1: Calculate threshold as the median of regression line values at midpoint of steepest windows
# IMPORTANT: Only include samples with substantial amplification (amplification_flags[col] == True)
# The threshold Y-value is calculated at the TRUE midpoint of the best-fit window.
# For a 4-cycle window starting at cycle 11 (cycles 11,12,13,14), midpoint = 12.5
midpoint_log_values = []
for col in columns_to_fit:
    if col not in steepest_windows or isinstance(steepest_windows[col], str):  # Skip samples flagged as having no exponential phase
        continue
    # Skip samples flagged as having no substantial amplification
    if not amplification_flags.get(col, True):
        print(f"Excluding {col} from threshold calculation (no substantial amplification)")
        continue
    start_cycle = steepest_windows[col]['start_cycle']  # 1-based cycle number
    slope = steepest_windows[col]['slope']
    intercept = steepest_windows[col]['intercept']
    # True midpoint: for window [11,12,13,14], midpoint = 11 + (4-1)/2 = 12.5
    midpoint_cycle = start_cycle + (window_size - 1) / 2
    midpoint_log = slope * midpoint_cycle + intercept  # Y-value at midpoint using regression line
    midpoint_log_values.append(midpoint_log)

if not midpoint_log_values:
    raise ValueError("No valid midpoints found for threshold calculation.")

threshold_log = np.median(midpoint_log_values)
threshold_linear = 10 ** threshold_log
print(f"Chosen threshold: Log10 = {threshold_log:.4f}, Linear = {threshold_linear:.4f}")

# Step 2: Calculate Cq using regression line crossing point
# Cq is where the 4-point best-fit line crosses the threshold in log space
for col in columns_to_fit:
    if col not in steepest_windows or isinstance(steepest_windows[col], str):  # Skip non-exponential-phase samples
        print(f"Skipping Cq calculation for {col}: No exponential phase identified.")
        continue
    # Skip samples flagged as having no substantial amplification
    if not amplification_flags.get(col, True):
        print(f"Skipping Cq calculation for {col}: No substantial amplification.")
        continue
    slope = steepest_windows[col]['slope']
    intercept = steepest_windows[col]['intercept']
    if slope == 0:
        print(f"Warning: Slope for {col} is zero. Skipping Cq calculation.")
        continue
    # Cq = cycle where regression line crosses threshold
    # Allow extrapolation beyond observed cycle range (may occur for late amplifying samples)
    cq = (threshold_log - intercept) / slope
    cq_values[col] = cq

# Step 3: Preview Cq values (only samples that passed Cell-3)
valid_columns = [col for col in columns_to_fit if amplification_flags.get(col, True)]
if cq_values and valid_columns:
    print("\nCq Values for Each Sample:")
    report_data = {
        'Sample': [],
        'Cq': []
    }
    for col in valid_columns:
        cq_val = cq_values.get(col, "N/A")
        report_data['Sample'].append(col)
        report_data['Cq'].append(f"{cq_val:.4f}" if isinstance(cq_val, (int, float)) else cq_val)
    report_df = pd.DataFrame(report_data)
    print(report_df.to_string(index=False))
else:
    print("No Cq values calculated for any samples.")

# Step 4: Export Cq values to CSV (Debug only)
if debug_flag and 'report_df' in locals() and not report_df.empty:
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    input_base_name = os.path.basename(file_path).replace('.csv', '')
    csv_filename = f"cq_values_{input_base_name}_{timestamp}.csv"
    csv_path = os.path.join(output_dir, csv_filename)
    with open(csv_path, 'w') as f:
        f.write(f"# Results from input file: {file_path}\n")
        f.write(f"# Generated on: {timestamp}\n")
        f.write(f"# Threshold (Log10): {threshold_log:.4f}\n")
        f.write(f"# Threshold (Linear): {threshold_linear:.4f}\n")
        f.write(f"# Cq Method: Regression line crossing point\n")
        report_df.to_csv(f, index=False)
    print(f"Saved (Debug/Evaluation) Cq values to {csv_path}")

# Step 5: Plot with threshold and regression lines
def plot_log_with_threshold(df_fine_tuned, df_log_refined, columns, threshold_log, steepest_windows, num_cycles, window_size, debug_flag=False, debug_display_flag=False, output_dir="outputs", file_path=""):
    """
    Plot log10-transformed fluorescence data with threshold and regression lines.
    """
    if not debug_display_flag and not debug_flag:
        print("Plot generation disabled. Skipping threshold plot.")
        return

    plt.figure(figsize=(12, 8), constrained_layout=True)
    cycles_full = np.arange(1, num_cycles + 1)
    colors = plt.cm.tab10(np.linspace(0, 1, len(columns)))

    for idx, column in enumerate(columns):
        adjusted_data = df_fine_tuned[column].iloc[:num_cycles].to_numpy()
        log_data = df_log_refined[column].iloc[:num_cycles].to_numpy()
        positive_mask = adjusted_data > 0
        cycles_positive = cycles_full[positive_mask]
        log_positive = log_data[positive_mask]

        if len(cycles_positive) > 0:
            plt.plot(cycles_positive, log_positive, label=f'{column}', marker='o', linestyle='-', color=colors[idx])
        else:
            print(f"Warning: No positive fluorescence values for {column}. Skipping plot.")

        # Overlay regression line for this sample's best window
        if column in steepest_windows and not isinstance(steepest_windows[column], str):
            start = steepest_windows[column]['start_cycle'] - 1
            window_cycles = np.arange(start + 1, start + window_size + 1)
            window_mask = (window_cycles >= cycles_positive[0]) & (window_cycles <= cycles_positive[-1])
            window_cycles_filtered = window_cycles[window_mask]

            if len(window_cycles_filtered) > 0:
                slope = steepest_windows[column]['slope']
                intercept = steepest_windows[column]['intercept']
                regression_line = slope * window_cycles_filtered + intercept
                cq_label = cq_values.get(column, None)
                if isinstance(cq_label, (int, float)):
                    cq_text = f"Cq={cq_label:.2f}"
                else:
                    cq_text = "Cq=N/A"
                plt.plot(window_cycles_filtered, regression_line, linestyle='--', color='red', linewidth=2,
                         label=f'{column} Best Window ({cq_text})')

    # Add threshold line
    plt.axhline(y=threshold_log, color='black', linestyle='--', linewidth=2, label=f'Threshold (Log10 = {threshold_log:.2f})')
    plt.xlabel('Cycle')
    plt.ylabel('Log10(Fluorescence)')
    plt.title(f'Log10-Transformed Refined Data with Threshold and Best Windows')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2 if len(columns) > 10 else 1)
    plt.grid(True)

    # Save the plot if requested (debug/evaluation)
    if debug_flag:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        input_base_name = os.path.basename(file_path).replace('.csv', '')
        plot_filename = f"log_with_threshold_{input_base_name}_{timestamp}.png"
        plot_path = os.path.join(output_dir, plot_filename)
        plt.savefig(plot_path, bbox_inches='tight')
        print(f"Saved (Debug/Evaluation) plot to {plot_path}")

    if debug_display_flag:
        plt.show()
        print("Plot displayed successfully.")
    else:
        plt.close()

# Plot the threshold and regression lines (only samples that passed Cell-3)
if valid_columns and steepest_windows and (debug_flag or eval_flag):
    plot_log_with_threshold(
        df_fine_tuned,
        df_log_refined,
        valid_columns,
        threshold_log,
        steepest_windows,
        num_cycles,
        window_size,
        debug_flag=(debug_flag or eval_flag),   # save plot in debug/eval
        debug_display_flag=debug_display_flag,  # display only in debug
        output_dir=output_dir,
        file_path=file_path
    )
else:
    if debug_flag or eval_flag:
        print("No columns or steepest windows to plot.")

# Outputs: threshold_log, threshold_linear, cq_values


In [ ]:
#==================  Cq and GF Combined - Cell-9  ======================
# Cell-9: Import max_val and KD from Cell-5 and generate seed guesses using Cq from Cell-8
"""
Purpose:
- Import refined max_val and KD from Cell-5's fitted_params.
- Generate seed guesses for each sample using Cq values and linear threshold from Cell-8.
- Use the formula: seed_guess = threshold_linear / (2 ** Cq), assuming ideal efficiency of 2.
- Report the data: sample, max_val, KD, seed_guess.

Inputs:
- fitted_params: Dictionary with final fitted parameters (from Cell-5).
- threshold_linear: The linear scale threshold from Cell-8.
- cq_values: Dictionary of Cq values from Cell-8.
- columns_to_fit: List of column names to process (from Cell-2).
- file_path: String, path to the qPCR data file (from Cell-1).
- debug_flag: True or False, indicating Debug mode (from Cell-1).
- eval_flag: True or False, indicating Evaluation mode (from Cell-1).
- output_dir: String, path to output directory (from Cell-1).

Outputs:
- Printed report for each sample: sample, max_val, KD, seed_guess.
- (Optional, Debug): Saved report (CSV).
"""

import pandas as pd
import numpy as np
import os
import datetime

# Verify inputs from Cell-5
try:
    fitted_params
except NameError:
    raise NameError("Required variable (fitted_params) not defined. Please run Cell-5 first.")

# Verify inputs from Cell-8
try:
    threshold_linear, cq_values
except NameError:
    raise NameError("Required variables (threshold_linear, cq_values) not defined. Please run Cell-8 first.")

# Verify inputs from Cell-7
try:
    amplification_flags
except NameError:
    raise NameError("Required variable (amplification_flags) not defined. Please run Cell-7 first.")

# Verify inputs from Cell-2
try:
    columns_to_fit
except NameError:
    raise NameError("Required variable (columns_to_fit) not defined. Please run Cell-2 first.")

# Verify inputs from Cell-1
try:
    file_path, debug_flag, eval_flag, output_dir
except NameError:
    raise NameError("file_path, debug_flag, eval_flag, or output_dir not defined. Please run Cell-1 first.")

print("Generating seed guesses for optimization...")

# Step 1: Report data for each sample (only those that passed Cell-3)
print("\n=== Sample Data: max_val, KD, and Seed Guess ===")
report_data = {
    'Sample': [],
    'max_val': [],
    'KD': [],
    'seed_guess': []
}

valid_columns = [col for col in columns_to_fit if amplification_flags.get(col, True)]
flagged_columns = [col for col in columns_to_fit if not amplification_flags.get(col, True)]

if flagged_columns:
    print(f"Skipped (no substantial amplification): {', '.join(flagged_columns)}")

for col in valid_columns:

    max_val = fitted_params.get(col, {}).get('final_max_val', 'N/A')
    KD = fitted_params.get(col, {}).get('final_KD', 'N/A')
    seed_guess = 'N/A'

    if col in cq_values and isinstance(cq_values[col], (int, float)):
        seed_guess = threshold_linear / (2 ** cq_values[col])
        print(f"{col}: max_val={max_val:.2f}, KD={KD:.2f}, seed_guess={seed_guess:.4e} (Cq={cq_values[col]:.2f})")
    else:
        print(f"{col}: max_val={max_val:.2f}, KD={KD:.2f}, seed_guess=N/A (no Cq available)")

    report_data['Sample'].append(col)
    report_data['max_val'].append(f"{max_val:.2f}" if isinstance(max_val, (int, float)) else max_val)
    report_data['KD'].append(f"{KD:.2f}" if isinstance(KD, (int, float)) else KD)
    report_data['seed_guess'].append(f"{seed_guess:.4e}" if isinstance(seed_guess, (int, float)) else seed_guess)

report_df = pd.DataFrame(report_data)
print("\nSummary Table:")
print(report_df.to_string(index=False))

# Step 2: Debug output: Save report to CSV
if debug_flag:
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    input_base_name = os.path.basename(file_path).replace('.csv', '')
    csv_filename = f"sample_data_report_{input_base_name}_{timestamp}.csv"
    csv_path = os.path.join(output_dir, csv_filename)
    report_df.to_csv(csv_path, index=False)
    print(f"\nSaved (Debug) report to {csv_path}")

# Step 3: Create seed_guesses dictionary for use in subsequent cells
seed_guesses = {}
for col in valid_columns:
    if col in cq_values and isinstance(cq_values[col], (int, float)):
        seed_guesses[col] = threshold_linear / (2 ** cq_values[col])
    else:
        seed_guesses[col] = None

print(f"\nSeed guesses generated for {len([s for s in seed_guesses.values() if s is not None])} samples.")
print("Seed guess range:", end=" ")
valid_seeds = [s for s in seed_guesses.values() if s is not None]
if valid_seeds:
    print(f"{min(valid_seeds):.4e} to {max(valid_seeds):.4e}")
else:
    print("No valid seed guesses")

# Outputs: report_df, seed_guesses (for potential use in further cells)


In [ ]:
#==================  Cq and GF Combined - Cell-10  ======================
# Cell-10: Optimize seed values using SOS minimization with PCR model propagation
"""
Purpose:
- Initialize seed_optimize values from seed_guess in Cell-9 for each sample.
- Compute modeled value for cycle 1 using seed and propagate values recursively up to the length of df_fine_tuned using the PCR model with refined max_val and KD from Cell-5.
- Optimize seed_optimized to minimize SOS between modeled and observed data.
- Plot observed data (black dots), initial model (blue dashed), and optimized model (green dashed) for visual comparison.
- Report max_val, KD, seed_guess, seed_optimized, initial SOS, and final SOS.

Inputs:
- fitted_params: Dictionary with final fitted parameters (from Cell-5).
- seed_guesses: Dictionary with seed guesses from Cell-9.
- df_fine_tuned: DataFrame with refined fluorescence data (from Cell-5).
- columns_to_fit: List of column names to process (from Cell-2).
- file_path: String, path to the qPCR data file (from Cell-1).
- debug_flag: True or False, indicating Debug mode (from Cell-1).
- debug_display_flag: True or False, indicating Debug plot display (from Cell-1).
- eval_flag: True or False, indicating Evaluation mode (from Cell-1).
- output_dir: String, path to output directory (from Cell-1).

Outputs:
- Printed report of max_val, KD, seed_guess, seed_optimized, initial SOS, and final SOS.
- Plots of observed data (black dots), initial model (blue dashed), and optimized model (green dashed) for each sample (Debug display only).
- (Optional, Debug): Saved report (CSV) and saved plots.
- (Optional, Debug/Evaluation): Global fitting plot (optimized models with observed data dots, single axes).
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import datetime
from scipy.optimize import minimize_scalar

# Verify inputs from Cell-5
try:
    fitted_params, df_fine_tuned
except NameError:
    raise NameError("Required variables (fitted_params, df_fine_tuned) not defined. Please run Cell-5 first.")

# Verify inputs from Cell-9
try:
    seed_guesses
except NameError:
    raise NameError("Required variable (seed_guesses) not defined. Please run Cell-9 first.")

# Verify inputs from Cell-7
try:
    amplification_flags
except NameError:
    raise NameError("Required variable (amplification_flags) not defined. Please run Cell-7 first.")

# Verify inputs from Cell-2
try:
    columns_to_fit
except NameError:
    raise NameError("Required variable (columns_to_fit) not defined. Please run Cell-2 first.")

# Verify inputs from Cell-1
try:
    file_path, debug_flag, debug_display_flag, eval_flag, output_dir
except NameError:
    raise NameError("file_path, debug_flag, debug_display_flag, eval_flag, or output_dir not defined. Please run Cell-1 first.")

# Evaluation/debug flags (Cell-1)
try:
    eval_flag
except NameError:
    eval_flag = False

try:
    debug_flag
except NameError:
    debug_flag = False

try:
    debug_display_flag
except NameError:
    debug_display_flag = False

print("Optimizing seed values using SOS minimization...")

# PCR model function
def pcr_model(prev, max_val, KD):
    """Single-step PCR model with non-negative constraint"""
    return np.maximum(0, prev * (1 + ((max_val - prev) / max_val) - (prev / (KD + prev))))

# Objective function for SOS minimization
def sos_objective(seed, max_val, KD, observed, n_cycles):
    """Compute SOS between modeled and observed data for a given seed."""
    # Generate full sequence starting from seed
    sequence = []
    prev = seed
    for i in range(n_cycles):
        if i == 0:
            # Cycle 1: use seed as previous value
            current = pcr_model(seed, max_val, KD)
        else:
            # Cycles 2-n: use previous cycle's result
            current = pcr_model(prev, max_val, KD)
        sequence.append(current)
        prev = current
    modeled = np.array(sequence)
    return np.sum((observed - modeled) ** 2)

# Logarithmic objective function for better scaling
def log_sos_objective(log_seed, max_val, KD, observed, n_cycles):
    """Compute SOS using logarithmic seed scaling for better optimization."""
    seed = 10 ** log_seed  # Convert from log space
    return sos_objective(seed, max_val, KD, observed, n_cycles)

# Step 1: Initialize and optimize seed values
print("\n=== Seed Optimization Results ===")
report_data = {
    'Sample': [],
    'max_val': [],
    'KD': [],
    'seed_guess': [],
    'initial_SOS': [],
    'seed_optimized': [],
    'final_SOS': [],
    'SOS_improvement': []
}

valid_columns = [col for col in columns_to_fit if amplification_flags.get(col, True)]
flagged_columns = [col for col in columns_to_fit if not amplification_flags.get(col, True)]

if flagged_columns:
    print(f"Skipped (no substantial amplification): {', '.join(flagged_columns)}")

# Include flagged samples in the report (as N/A), but do not optimize them
for col in columns_to_fit:
    if not amplification_flags.get(col, True):
        report_data['Sample'].append(col)
        report_data['max_val'].append('N/A')
        report_data['KD'].append('N/A')
        report_data['seed_guess'].append('N/A')
        report_data['initial_SOS'].append('N/A')
        report_data['seed_optimized'].append('N/A (no substantial amplification)')
        report_data['final_SOS'].append('N/A')
        report_data['SOS_improvement'].append('N/A')

for col in valid_columns:
    print(f"\nProcessing {col}...")

    # Get parameters
    max_val = fitted_params.get(col, {}).get('final_max_val', None)
    KD = fitted_params.get(col, {}).get('final_KD', None)
    seed_guess = seed_guesses.get(col, None)

    if max_val is None or KD is None or seed_guess is None:
        print(f"Warning: Missing parameters for {col}. Skipping.")
        report_data['Sample'].append(col)
        report_data['max_val'].append('N/A')
        report_data['KD'].append('N/A')
        report_data['seed_guess'].append('N/A')
        report_data['initial_SOS'].append('N/A')
        report_data['seed_optimized'].append('N/A')
        report_data['final_SOS'].append('N/A')
        report_data['SOS_improvement'].append('N/A')
        continue

    # Get observed data
    observed = df_fine_tuned[col].values.copy()
    n_cycles = len(observed)

    # Calculate initial SOS
    initial_sos = sos_objective(seed_guess, max_val, KD, observed, n_cycles)
    print(f"  Initial SOS: {initial_sos:.4e}")

    # Optimize seed using logarithmic scaling for better handling of large dynamic ranges
    try:
        # Convert seed guess to log space for optimization
        log_seed_guess = np.log10(seed_guess)

        # Set bounds in log space (factor of 3 in each direction = 1000x range)
        log_bounds_factor = 3.0
        log_lower_bound = log_seed_guess - log_bounds_factor
        log_upper_bound = log_seed_guess + log_bounds_factor

        # Try logarithmic optimization
        result = minimize_scalar(
            log_sos_objective,
            args=(max_val, KD, observed, n_cycles),
            method='bounded',
            bounds=(log_lower_bound, log_upper_bound),
            options={'xatol': 1e-4, 'maxiter': 100}
        )

        if result.success:
            log_seed_optimized = result.x
            seed_optimized = 10 ** log_seed_optimized  # Convert back to linear space
            final_sos = result.fun
            sos_improvement = ((initial_sos - final_sos) / initial_sos) * 100

            # Check if optimization actually improved the fit
            if final_sos < initial_sos:
                print(f"  Optimized seed: {seed_optimized:.4e} (log: {log_seed_optimized:.4f})")
                print(f"  Final SOS: {final_sos:.4e}")
                print(f"  SOS improvement: {sos_improvement:.2f}%")
            else:
                print(f"  Optimization found worse fit, using seed guess")
                seed_optimized = seed_guess
                final_sos = initial_sos
                sos_improvement = 0.0
        else:
            print(f"  Logarithmic optimization failed: {result.message}")
            # Fall back to seed guess
            seed_optimized = seed_guess
            final_sos = initial_sos
            sos_improvement = 0.0

    except Exception as e:
        print(f"  Optimization error: {e}")
        seed_optimized = seed_guess
        final_sos = initial_sos
        sos_improvement = 0.0

    # Store results
    report_data['Sample'].append(col)
    report_data['max_val'].append(f"{max_val:.2f}")
    report_data['KD'].append(f"{KD:.2f}")
    report_data['seed_guess'].append(f"{seed_guess:.4e}")
    report_data['initial_SOS'].append(f"{initial_sos:.4e}")
    report_data['seed_optimized'].append(f"{seed_optimized:.4e}")
    report_data['final_SOS'].append(f"{final_sos:.4e}")
    report_data['SOS_improvement'].append(f"{sos_improvement:.2f}%")

# Create report DataFrame
report_df = pd.DataFrame(report_data)
print("\n=== Optimization Summary ===")
print(report_df.to_string(index=False))

# Step 2: Generate plots for each sample
def plot_seed_optimization(col, max_val, KD, seed_guess, seed_optimized, observed, debug_flag=False, debug_display_flag=False, output_dir="outputs", file_path=""):
    """Plot observed vs modeled data for seed optimization"""
    if not debug_display_flag and not debug_flag:
        print(f"Debug output disabled. Skipping plot for {col}.")
        return

    n_cycles = len(observed)
    cycles = np.arange(1, n_cycles + 1)

    # Generate initial model sequence
    initial_sequence = []
    prev = seed_guess
    for i in range(n_cycles):
        if i == 0:
            current = pcr_model(seed_guess, max_val, KD)
        else:
            current = pcr_model(prev, max_val, KD)
        initial_sequence.append(current)
        prev = current

    # Generate optimized model sequence
    optimized_sequence = []
    prev = seed_optimized
    for i in range(n_cycles):
        if i == 0:
            current = pcr_model(seed_optimized, max_val, KD)
        else:
            current = pcr_model(prev, max_val, KD)
        optimized_sequence.append(current)
        prev = current

    # Create plot
    fig, ax = plt.subplots(figsize=(12, 8))
    plt.plot(cycles, observed, 'ko', label='Observed Data', markersize=4)
    plt.plot(cycles, initial_sequence, 'b--', label=f'Initial Model (seed={seed_guess:.2e})', linewidth=2)
    plt.plot(cycles, optimized_sequence, 'g--', label=f'Optimized Model (seed={seed_optimized:.2e})', linewidth=2)
    plt.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    plt.xlabel('Cycle')
    plt.ylabel('Fluorescence Value')
    plt.title(f'{col}: Seed Optimization Results\nmax_val={max_val:.2f}, KD={KD:.2f}')
    plt.legend()
    plt.grid(True)

    # Save plot in Debug mode
    if debug_flag:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        input_base_name = os.path.basename(file_path).replace('.csv', '')
        plot_filename = f"seed_optimization_{col}_{input_base_name}_{timestamp}.png"
        plot_path = os.path.join(output_dir, plot_filename)
        plt.savefig(plot_path, bbox_inches='tight')
        print(f"Saved (Debug) plot to {plot_path}")

    if debug_display_flag:
        plt.show()
        print(f"Plot displayed for {col}.")
    else:
        plt.close()

# Generate plots for each sample (only those that passed Cell-3)
for col in valid_columns:

    max_val = fitted_params.get(col, {}).get('final_max_val', None)
    KD = fitted_params.get(col, {}).get('final_KD', None)
    seed_guess = seed_guesses.get(col, None)
    seed_optimized = report_df.loc[report_df['Sample'] == col, 'seed_optimized'].iloc[0]

    if max_val is not None and KD is not None and seed_guess is not None:
        # Convert seed_optimized back to float
        try:
            seed_optimized = float(seed_optimized)
        except:
            seed_optimized = seed_guess

        observed = df_fine_tuned[col].values
        plot_seed_optimization(col, max_val, KD, seed_guess, seed_optimized, observed, debug_flag, debug_display_flag, output_dir, file_path)

# Step 3: Debug output: Save report to CSV (only valid samples)
if debug_flag and report_data['Sample']:
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    input_base_name = os.path.basename(file_path).replace('.csv', '')
    csv_filename = f"seed_optimization_results_{input_base_name}_{timestamp}.csv"
    csv_path = os.path.join(output_dir, csv_filename)
    report_df.to_csv(csv_path, index=False)
    print(f"\nSaved (Debug) optimization results to {csv_path}")

# Step 4: Create final seed_optimized dictionary for use in subsequent cells
seed_optimized_dict = {}
for col in valid_columns:
    if col in report_df['Sample'].values:
        seed_opt = report_df.loc[report_df['Sample'] == col, 'seed_optimized'].iloc[0]
        try:
            seed_optimized_dict[col] = float(seed_opt)
        except:
            seed_optimized_dict[col] = seed_guesses.get(col, None)

print(f"\nSeed optimization completed for {len([s for s in seed_optimized_dict.values() if s is not None])} samples.")

# Step 5 (Evaluation/Debug): Plot global fitting models (optimized only)
def plot_global_fitting_models(seed_optimized_dict, fitted_params, df_fine_tuned, output_dir="outputs", file_path="", save_plot=False, show_plot=False):
    """Plot all optimized model curves on a single axes with observed data dots."""
    if not seed_optimized_dict:
        print("No optimized seeds available for global fitting plot.")
        return

    fig, ax = plt.subplots(figsize=(12, 8))
    for col, seed_opt in seed_optimized_dict.items():
        if seed_opt is None:
            continue
        max_val = fitted_params.get(col, {}).get('final_max_val', None)
        KD = fitted_params.get(col, {}).get('final_KD', None)
        if max_val is None or KD is None:
            continue

        n_cycles = len(df_fine_tuned[col].values)
        cycles = np.arange(1, n_cycles + 1)
        observed = df_fine_tuned[col].values
        modeled = []
        prev = seed_opt
        for i in range(n_cycles):
            if i == 0:
                current = pcr_model(seed_opt, max_val, KD)
            else:
                current = pcr_model(prev, max_val, KD)
            modeled.append(current)
            prev = current
        ax.plot(cycles, observed, 'ko', markersize=3, alpha=0.6, label=f"{col} Observed")
        ax.plot(cycles, modeled, linewidth=2, alpha=0.9, label=f"{col} Model")

    ax.set_xlabel('Cycle')
    ax.set_ylabel('Fluorescence Value')
    ax.set_title('Global Fitting Plot (Optimized Models + Observed)')
    ax.grid(True)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.subplots_adjust(right=0.75)

    if save_plot:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        input_base_name = os.path.basename(file_path).replace('.csv', '')
        plot_filename = f"Global_fitting_plot_{input_base_name}_{timestamp}.png"
        plot_path = os.path.join(output_dir, plot_filename)
        fig.savefig(plot_path, bbox_inches='tight')
        print(f"Saved (Debug/Evaluation) global fitting plot to {plot_path}")

    if show_plot:
        plt.show()
        print("Global fitting plot displayed.")
    else:
        plt.close(fig)

if (eval_flag or debug_flag):
    plot_global_fitting_models(
        seed_optimized_dict,
        fitted_params,
        df_fine_tuned,
        output_dir=output_dir,
        file_path=file_path,
        save_plot=True,
        show_plot=debug_display_flag
    )

# Step 6 (Debug): Export optimized seed-modeled data
if debug_flag and seed_optimized_dict:
    cycles = np.arange(1, len(df_fine_tuned) + 1)
    modeled_df = pd.DataFrame({'Cycle': cycles})
    for col, seed_opt in seed_optimized_dict.items():
        if seed_opt is None:
            continue
        max_val = fitted_params.get(col, {}).get('final_max_val', None)
        KD = fitted_params.get(col, {}).get('final_KD', None)
        if max_val is None or KD is None:
            continue
        modeled = []
        prev = seed_opt
        for i in range(len(cycles)):
            if i == 0:
                current = pcr_model(seed_opt, max_val, KD)
            else:
                current = pcr_model(prev, max_val, KD)
            modeled.append(current)
            prev = current
        modeled_df[col] = modeled

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    input_base_name = os.path.basename(file_path).replace('.csv', '')
    modeled_filename = f"seed_modeled_data_{input_base_name}_{timestamp}.csv"
    modeled_path = os.path.join(output_dir, modeled_filename)
    modeled_df.to_csv(modeled_path, index=False)
    print(f"Saved (Debug) seed-modeled data to {modeled_path}")

# Outputs: report_df, seed_optimized_dict


In [ ]:
#==================  Cq and GF Combined - Cell-11  ======================
# Cell-11: Generate final analysis outputs for downstream plotting or analyses (e.g., delta methods)
"""
Purpose:
- Consolidate all key analysis results into a single CSV file for downstream analysis.
 - Include Cq values, seed, final max, final KD, and max/KD ratio.
- Generate clean output suitable for spreadsheet analysis and delta method calculations.
- Always display results table and summary statistics (easy to copy from notebook or command line).

Inputs:
- cq_values: Dictionary of Cq values (from Cell-8).
- seed_optimized_dict: Dictionary of seed values (from Cell-10).
- fitted_params: Dictionary with final fitted parameters (from Cell-5).
- columns_to_fit: List of column names to process (from Cell-2).
- file_path: String, path to the qPCR data file (from Cell-1).
- debug_flag: True or False, indicating Debug mode (from Cell-1).
- eval_flag: True or False, indicating Evaluation mode (from Cell-1).
- output_dir: String, path to output directory (from Cell-1).

Outputs:
- Final CSV file: 'input_file_name_qPCR_Analysis_Outputs_datestamp.csv'
- Printed results table and summary statistics (always displayed for easy copy/reference)
"""

import pandas as pd
import numpy as np
import os
import datetime

# Verify inputs from Cell-8
try:
    cq_values
except NameError:
    raise NameError("Required variable (cq_values) not defined. Please run Cell-8 first.")

# Verify inputs from Cell-10
try:
    seed_optimized_dict
except NameError:
    raise NameError("Required variable (seed_optimized_dict) not defined. Please run Cell-10 first.")

# Verify inputs from Cell-5
try:
    fitted_params
except NameError:
    raise NameError("Required variable (fitted_params) not defined. Please run Cell-5 first.")

# Verify inputs from Cell-7
try:
    amplification_flags
except NameError:
    raise NameError("Required variable (amplification_flags) not defined. Please run Cell-7 first.")

# Verify inputs from Cell-2
try:
    columns_to_fit
except NameError:
    raise NameError("Required variable (columns_to_fit) not defined. Please run Cell-2 first.")

# Verify inputs from Cell-1
try:
    file_path, debug_flag, eval_flag, output_dir
except NameError:
    raise NameError("file_path, debug_flag, eval_flag, or output_dir not defined. Please run Cell-1 first.")

print("Generating final analysis outputs...")

# Step 1: Compile final results
final_results = {
    'Sample': [],
    'Amplification_Status': [],
    'Cq': [],
    'Seed': [],
    'Max': [],
    'KD': [],
    'Max_KD_Ratio': []
}

for col in columns_to_fit:
    # Check amplification status
    has_amplification = amplification_flags.get(col, True)

    if not has_amplification:
        # Sample flagged as no substantial amplification
        final_results['Sample'].append(col)
        final_results['Amplification_Status'].append('No substantial amplification')
        final_results['Cq'].append('N/A')
        final_results['Seed'].append('N/A')
        # Still report max_val and KD from model fit (useful for documentation)
        max_val = fitted_params.get(col, {}).get('final_max_val', 'N/A')
        KD = fitted_params.get(col, {}).get('final_KD', 'N/A')
        final_results['Max'].append(f"{max_val:.2f}" if isinstance(max_val, (int, float)) else max_val)
        final_results['KD'].append(f"{KD:.2f}" if isinstance(KD, (int, float)) else KD)
        final_results['Max_KD_Ratio'].append('N/A')
        continue

    # Get Cq value
    cq_val = cq_values.get(col, 'N/A')

    # Get optimized seed
    seed_opt = seed_optimized_dict.get(col, 'N/A')

    # Get final parameters
    max_val = fitted_params.get(col, {}).get('final_max_val', 'N/A')
    KD = fitted_params.get(col, {}).get('final_KD', 'N/A')

    # Calculate max_val/KD ratio
    if isinstance(max_val, (int, float)) and isinstance(KD, (int, float)) and KD != 0:
        ratio = max_val / KD
    else:
        ratio = 'N/A'

    # Add to results
    final_results['Sample'].append(col)
    final_results['Amplification_Status'].append('Amplified')
    final_results['Cq'].append(f"{cq_val:.4f}" if isinstance(cq_val, (int, float)) else cq_val)
    final_results['Seed'].append(f"{seed_opt:.4e}" if isinstance(seed_opt, (int, float)) else seed_opt)
    final_results['Max'].append(f"{max_val:.2f}" if isinstance(max_val, (int, float)) else max_val)
    final_results['KD'].append(f"{KD:.2f}" if isinstance(KD, (int, float)) else KD)
    final_results['Max_KD_Ratio'].append(f"{ratio:.4f}" if isinstance(ratio, (int, float)) else ratio)

# Create final DataFrame
final_df = pd.DataFrame(final_results)

# Step 2: Always display final results (easy to copy from notebook or command line)
print("\n=== Final qPyCR Analysis Results ===")
print("Results suitable for subsequent calculations:")
print(final_df.to_string(index=False))

# Summary statistics (always shown for quick reference)
print("\n=== Summary Statistics ===")
valid_cq = [float(x) for x in final_df['Cq'] if x != 'N/A']
valid_seeds = [float(x) for x in final_df['Seed'] if x != 'N/A']
valid_ratios = [float(x) for x in final_df['Max_KD_Ratio'] if x != 'N/A']

if valid_cq:
    print(f"Cq - Range: {min(valid_cq):.2f} to {max(valid_cq):.2f}, Mean: {np.mean(valid_cq):.2f}")
if valid_seeds:
    print(f"Seed Optimized - Range: {min(valid_seeds):.4e} to {max(valid_seeds):.4e}")
if valid_ratios:
    print(f"Max/KD Ratio - Range: {min(valid_ratios):.2f} to {max(valid_ratios):.2f}, Mean: {np.mean(valid_ratios):.2f}")

# Step 3: Generate final CSV file
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
input_base_name = os.path.basename(file_path).replace('.csv', '')
csv_filename = f"{input_base_name}_qPCR_Analysis_Outputs_{timestamp}.csv"
csv_path = os.path.join(output_dir, csv_filename)

# Count amplified vs non-amplified samples
amplified_count = len([col for col in columns_to_fit if amplification_flags.get(col, True)])
non_amplified_count = len(columns_to_fit) - amplified_count

# Add metadata header
with open(csv_path, 'w') as f:
    f.write(f"# qPCR Analysis Results\n")
    f.write(f"# Input file: {file_path}\n")
    f.write(f"# Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"# Analysis method: Custom qPCR pipeline with recursive PCR model\n")
    f.write(f"# Samples with substantial amplification: {amplified_count}\n")
    f.write(f"# Samples with no substantial amplification: {non_amplified_count}\n")
    f.write(f"# Columns: Sample, Amplification_Status, Cq, Seed, Max, KD, Max_KD_Ratio\n")
    f.write(f"# Use for delta method calculations and comparative analysis\n")
    f.write(f"#\n")
    final_df.to_csv(f, index=False)

print(f"\nFinal analysis results saved to: {csv_path}")
print(f"File contains {len(final_df)} samples with complete analysis results.")
print("Results are ready for delta method calculations and comparative analysis.")

# Outputs: final_df, csv_path
